# 02 — PostgreSQL Data Pipeline
## Climate Change: Earth Surface Temperature Analysis

**Người thực hiện:** Cao Tấn Phát — Data Engineer  
**Vai trò của notebook:** chuyển năm CSV đã được khảo sát trong `01_data_understanding.ipynb` thành một lớp dữ liệu PostgreSQL có cấu trúc, có kiểm tra và có thể truy vết.

Notebook này nằm giữa bước hiểu dữ liệu và bước làm sạch. Nó không thay thế kết luận của Notebook 01 và không tự quyết định cách điền missing values hoặc xử lý outlier. Đầu ra chi tiết theo tháng (`vw_*_temperature`) là dữ liệu bàn giao chính cho `03_data_cleaning.ipynb`; các materialized views theo năm/thập kỷ là dữ liệu tổng hợp tham khảo cho phân tích sau đó.

## 1. Mục tiêu và kiến trúc pipeline

Mục 1 giải thích notebook nhận dữ liệu nào, bảo toàn điều gì và tạo ra những đối tượng PostgreSQL nào. Đây là bản đồ để người đọc hiểu toàn bộ pipeline trước khi chạy các cell có tác động đến database.

Notebook 01 chịu trách nhiệm khảo sát cấu trúc, missing values và duplicates của CSV. Notebook 02 sử dụng các kết quả đó như data contract, nạp nguyên trạng vào staging, chuẩn hóa khóa/ngày/tọa độ rồi kiểm tra dữ liệu giữa các tầng. Notebook 03 chỉ nên bắt đầu cleaning sau khi các validation ở Notebook 02 đều `PASS`.

> **Phạm vi của Mục 1:** chỉ mô tả kiến trúc và khai báo đường dẫn/tên dataset dùng chung; chưa kết nối PostgreSQL, chưa tạo bảng và chưa thay đổi dữ liệu.

### a. Mục tiêu

Notebook xây dựng luồng ETL có thể chạy lại cho năm cấp dữ liệu: **Global, Country, State, City** và **Major City**. Một lần chạy hoàn chỉnh cần đạt các mục tiêu sau:

1. Đọc thông tin kết nối từ biến môi trường, không ghi mật khẩu trong notebook.
2. Xác nhận các CSV vẫn đúng file/header/row count đã khảo sát trong Notebook 01.
3. Nạp dữ liệu thô vào staging bằng `COPY FROM STDIN` và đối chiếu số dòng.
4. Chuẩn hóa ngày và khóa địa lý thành dimension/fact nhưng giữ nguyên temperature `NULL`.
5. Tạo monthly analytical views, kiểm tra join và phân biệt khoảng trống nguồn với lỗi pipeline.
6. Tạo aggregation theo năm/thập kỷ và index phục vụ truy vấn.
7. Cung cấp kết quả validation cùng dữ liệu có lineage để Notebook 03 tiếp tục cleaning.

Dữ liệu trong `data/raw/` luôn chỉ đọc. Notebook này không tạo `data/processed`; các file đã làm sạch chỉ được tạo bởi những notebook tiếp theo sau khi quy tắc cleaning được thống nhất.

### b. Luồng dữ liệu, cấp độ quan sát và thứ tự xử lý

```text
data/raw/*.csv
      ↓
Xác nhận data contract từ Notebook 01: file, header, row count
      ↓
PostgreSQL staging tables: giữ nguyên cấu trúc gần với CSV gốc
      ↓
Chuẩn hóa ngày, khóa địa lý và tọa độ ở tầng phân tích
      ↓
Dimension / fact tables → monthly analytical views
      ↓
Join theo `dt` và khóa địa lý phù hợp
      ↓
Join validation → materialized aggregations → indexes/query plans
```

Năm CSV có grain khác nhau: Global–tháng, Country–tháng, State–tháng, City–tháng và Major City–tháng. Pipeline giữ riêng năm fact tables để không làm sai grain hoặc nhân bản dữ liệu. Các quan hệ giữa cấp địa lý chỉ được dùng trong truy vấn có khóa thời gian và geography rõ ràng.

Lớp staging gồm `staging_global`, `staging_country`, `staging_state`, `staging_city` và `staging_major_city`. Lớp phân tích gồm bốn dimensions, năm fact tables, năm monthly views và sáu materialized aggregation views. `source_staging_id` được giữ trong fact/view để Notebook 03 có thể truy ngược bản ghi nguồn khi điều tra dữ liệu.

Thứ tự thực thi là `01_create_tables.sql` → `COPY FROM STDIN` → row-count validation → dimension/fact → `03_views.sql` → join validation → `04_aggregation.sql` → `05_indexes.sql`. `02_import_data.sql` tài liệu hóa mapping cột; notebook điều khiển luồng STDIN để PostgreSQL cục bộ không phải truy cập trực tiếp đường dẫn CSV.

In [1]:
# Cấu hình mô tả pipeline: chưa kết nối PostgreSQL và chưa đọc dữ liệu CSV.
from pathlib import Path


# Dò từ thư mục hiện tại lên các thư mục cha để notebook chạy được từ root hoặc notebooks/.
def find_project_root(start: Path) -> Path:
    """Tìm thư mục gốc chứa cả data/raw và SQL khi chạy notebook ở các vị trí khác nhau."""
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'raw').is_dir() and (candidate / 'SQL').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy project root chứa data/raw và SQL.')


# Các đường dẫn dùng chung được tạo một lần để các mục sau không hard-code đường dẫn máy.
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
SQL_DIR = PROJECT_ROOT / 'SQL'

# Mỗi dataset được gắn với file nguồn, staging table và grain nghiệp vụ tương ứng.
DATASET_CONFIG = {
    'global': {
        'filename': 'GlobalTemperatures.csv',
        'staging_table': 'staging_global',
        'grain': 'một quan sát toàn cầu mỗi tháng',
    },
    'country': {
        'filename': 'GlobalLandTemperaturesByCountry.csv',
        'staging_table': 'staging_country',
        'grain': 'một quan sát quốc gia mỗi tháng',
    },
    'state': {
        'filename': 'GlobalLandTemperaturesByState.csv',
        'staging_table': 'staging_state',
        'grain': 'một quan sát bang/tỉnh mỗi tháng',
    },
    'city': {
        'filename': 'GlobalLandTemperaturesByCity.csv',
        'staging_table': 'staging_city',
        'grain': 'một quan sát thành phố mỗi tháng',
    },
    'major_city': {
        'filename': 'GlobalLandTemperaturesByMajorCity.csv',
        'staging_table': 'staging_major_city',
        'grain': 'một quan sát thành phố lớn mỗi tháng',
    },
}

# Thứ tự này phản ánh dependency: tables → views → aggregations → indexes.
SQL_EXECUTION_ORDER = (
    '01_create_tables.sql',
    '03_views.sql',
    '04_aggregation.sql',
    '05_indexes.sql',
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Datasets planned for staging: {len(DATASET_CONFIG)}')
for name, config in DATASET_CONFIG.items():
    print(f"- {name}: {config['filename']} → {config['staging_table']}")

Project root: E:\FPT\HocKy3\PROJECT_1\PROJECT\Global-Surface-Temperature-Analysis
Datasets planned for staging: 5
- global: GlobalTemperatures.csv → staging_global
- country: GlobalLandTemperaturesByCountry.csv → staging_country
- state: GlobalLandTemperaturesByState.csv → staging_state
- city: GlobalLandTemperaturesByCity.csv → staging_city
- major_city: GlobalLandTemperaturesByMajorCity.csv → staging_major_city


## 2. Imports và cấu hình `.env`

Mục này chuẩn bị thư viện Python và tạo `DB_CONFIG` từ file `.env` ở project root. Tách cấu hình khỏi code giúp thành viên khác chạy cùng notebook với PostgreSQL của họ mà không sửa cell hoặc làm lộ mật khẩu.

Các biến bắt buộc là `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`; `DB_SSLMODE` là tùy chọn. Mật khẩu chỉ tồn tại trong bộ nhớ kernel, không được in ra output và `.env` phải tiếp tục nằm trong `.gitignore`.

**Đầu ra của mục:** dictionary `DB_CONFIG` dùng cho tất cả kết nối sau đó. Cell chưa truy vấn database và không liên quan đến nội dung dữ liệu mà Notebook 01 đã khảo sát.

In [2]:
# Các import dùng chung cho PostgreSQL pipeline.
import os
from pathlib import Path

import pandas as pd
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv


# Chỉ đọc credentials từ .env ở project root; không ghi mật khẩu trực tiếp vào notebook.
ENV_PATH = PROJECT_ROOT / '.env'
if not ENV_PATH.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy file .env tại: {ENV_PATH}. '
        'Hãy tạo file .env ở thư mục gốc dự án trước khi tiếp tục.'
    )

load_dotenv(dotenv_path=ENV_PATH, override=False)

# Fail fast nếu thiếu cấu hình để lỗi kết nối phía sau có thông báo dễ hiểu hơn.
REQUIRED_DB_ENV = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
missing_variables = [name for name in REQUIRED_DB_ENV if not os.getenv(name)]
if missing_variables:
    raise EnvironmentError(
        'Thiếu biến môi trường bắt buộc trong .env: ' + ', '.join(missing_variables)
    )

try:
    db_port = int(os.environ['DB_PORT'])
except ValueError as error:
    raise ValueError('DB_PORT phải là một số nguyên, ví dụ 5432.') from error

# Chuẩn hóa các biến môi trường sang dictionary mà psycopg2 chấp nhận.
DB_CONFIG = {
    'host': os.environ['DB_HOST'],
    'port': db_port,
    'dbname': os.environ['DB_NAME'],
    'user': os.environ['DB_USER'],
    'password': os.environ['DB_PASSWORD'],
    'connect_timeout': 10,
}

if os.getenv('DB_SSLMODE'):
    DB_CONFIG['sslmode'] = os.environ['DB_SSLMODE']

print(f'Đã nạp cấu hình từ: {ENV_PATH.name}')
print(f"Database target: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
print(f"Database user: {DB_CONFIG['user']}")
print('Mật khẩu đã được nạp nhưng không hiển thị trong output.')

Đã nạp cấu hình từ: .env
Database target: localhost:5432/climate_db
Database user: postgres
Mật khẩu đã được nạp nhưng không hiển thị trong output.


## 3. Kiểm tra kết nối PostgreSQL

Mục này mở một kết nối ngắn bằng `DB_CONFIG` và đọc tên database, user cùng phiên bản PostgreSQL. Đây là checkpoint hạ tầng: nếu thất bại thì các bước ETL chưa được phép chạy, nhưng lỗi này không nói lên CSV đúng hay sai.

Truy vấn hoàn toàn chỉ đọc và connection được đóng ngay sau kiểm tra. Nếu lỗi, hãy kiểm tra PostgreSQL service, host/port, database `climate_db`, user và `.env`; không cần chạy lại Notebook 01 hoặc thay đổi CSV.

**Đầu ra của mục:** bằng chứng notebook đang kết nối đúng database trước khi tạo schema.

In [3]:
# Health check chỉ đọc: xác nhận kết nối và đóng connection sau khi hoàn tất.
from contextlib import closing

# Truy vấn nhỏ này xác nhận notebook đang trỏ đúng database/user trước khi ghi dữ liệu.
try:
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                'SELECT current_database(), current_user, version();'
            )
            database_name, database_user, server_version = cursor.fetchone()
except psycopg2.Error as error:
    raise ConnectionError(
        'Không thể kết nối PostgreSQL. Hãy kiểm tra PostgreSQL service và cấu hình .env.'
    ) from error

print('Kết nối PostgreSQL thành công.')
print(f'Database: {database_name}')
print(f'User: {database_user}')
print(f'Server: {server_version.splitlines()[0]}')

Kết nối PostgreSQL thành công.
Database: climate_db
User: postgres
Server: PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit


## 4. Xác nhận dữ liệu đầu vào trước import

Mục này là điểm bàn giao trực tiếp từ `01_data_understanding.ipynb`. Notebook 01 đã đọc toàn bộ dữ liệu để khảo sát schema, missing values, duplicates và row count; Notebook 02 không lặp lại công việc đó. Thay vào đó, cell chỉ xác nhận đúng năm file, file không rỗng và header chưa thay đổi.

Các row count được Notebook 01 xác nhận được ghi vào `SOURCE_ROW_COUNTS` như một data contract. Mục 6–8 sẽ dùng chúng để phát hiện import thiếu hoặc transformation làm mất dòng. Nếu CSV bị thay thế, đổi header hoặc đổi số dòng, phải quay lại Notebook 01, cập nhật nhận xét và data contract trước khi chạy tiếp.

**Đầu ra của mục:** `input_check_df` và `SOURCE_ROW_COUNTS`. Notebook 03 có thể tin rằng dữ liệu PostgreSQL xuất phát đúng từ bộ CSV đã được khảo sát khi toàn bộ kiểm tra đầu vào hợp lệ.

In [4]:
# Khai báo data contract của 5 CSV đã được khảo sát trong Notebook 01.
import csv

from IPython.display import display


# Thứ tự cột cũng được kiểm tra vì COPY ánh xạ theo vị trí đã khai báo.
EXPECTED_CSV_COLUMNS = {
    'global': [
        'dt', 'LandAverageTemperature', 'LandAverageTemperatureUncertainty',
        'LandMaxTemperature', 'LandMaxTemperatureUncertainty',
        'LandMinTemperature', 'LandMinTemperatureUncertainty',
        'LandAndOceanAverageTemperature',
        'LandAndOceanAverageTemperatureUncertainty',
    ],
    'country': [
        'dt', 'AverageTemperature', 'AverageTemperatureUncertainty', 'Country',
    ],
    'state': [
        'dt', 'AverageTemperature', 'AverageTemperatureUncertainty',
        'State', 'Country',
    ],
    'city': [
        'dt', 'averagetemperature', 'averagetemperatureuncertainty',
        'city', 'country', 'latitude', 'longitude',
    ],
    'major_city': [
        'dt', 'AverageTemperature', 'AverageTemperatureUncertainty',
        'City', 'Country', 'Latitude', 'Longitude',
    ],
}

# Row count đã được xác nhận trong 01_data_understanding.ipynb.
SOURCE_ROW_COUNTS = {
    'global': 3_192,
    'country': 577_462,
    'state': 645_675,
    'city': 5_010_113,
    'major_city': 239_177,
}

In [5]:
# Kiểm tra nhẹ metadata và header, không đọc toàn bộ các CSV lớn vào RAM.
input_checks = []

for dataset_name, config in DATASET_CONFIG.items():
    csv_path = RAW_DATA_DIR / config['filename']
    file_exists = csv_path.is_file()
    file_size_mb = csv_path.stat().st_size / (1024 ** 2) if file_exists else 0.0
    actual_header = []

    # utf-8-sig loại bỏ BOM nếu file có BOM ở đầu header.
    if file_exists and csv_path.stat().st_size > 0:
        with csv_path.open('r', encoding='utf-8-sig', newline='') as csv_file:
            actual_header = next(csv.reader(csv_file), [])

    expected_header = EXPECTED_CSV_COLUMNS[dataset_name]
    input_checks.append({
        'dataset': dataset_name,
        'file': config['filename'],
        'exists': file_exists,
        'size_mb': file_size_mb,
        'header_ok': actual_header == expected_header,
        'expected_rows': SOURCE_ROW_COUNTS[dataset_name],
    })

input_check_df = pd.DataFrame(input_checks)
display(input_check_df)

# Dừng pipeline ngay khi file thiếu, rỗng hoặc header lệch khỏi data contract.
invalid_inputs = input_check_df.loc[
    (~input_check_df['exists'])
    | (input_check_df['size_mb'] <= 0)
    | (~input_check_df['header_ok'])
]

if not invalid_inputs.empty:
    raise ValueError(
        'Dữ liệu đầu vào không còn khớp với kết quả Notebook 01. '
        'Hãy kiểm tra file hoặc chạy lại 01_data_understanding.ipynb.'
    )

print('Đã xác nhận đủ 5 CSV và header khớp với Notebook 01.')
print(f'Tổng row count dự kiến sau import: {sum(SOURCE_ROW_COUNTS.values()):,}')

,dataset,file,exists,size_mb,header_ok,expected_rows
0,global,GlobalTemperatures.csv,True,0.196338,True,3192
1,country,GlobalLandTemperaturesByCountry.csv,True,21.629708,True,577462
2,state,GlobalLandTemperaturesByState.csv,True,29.344711,True,645675
3,city,GlobalLandTemperaturesByCity.csv,True,297.837700,True,5010113
4,major_city,GlobalLandTemperaturesByMajorCity.csv,True,13.483415,True,239177


Đã xác nhận đủ 5 CSV và header khớp với Notebook 01.
Tổng row count dự kiến sau import: 6,475,619


## 5. Tạo lại các bảng staging

Mục này tạo điểm tiếp nhận dữ liệu thô trong PostgreSQL. `SQL/01_create_tables.sql` xóa schema cũ bằng `DROP TABLE IF EXISTS ... CASCADE` rồi tạo lại năm staging tables với cấu trúc gần CSV, khóa `staging_id` và thời điểm `loaded_at`. Staging giữ dữ liệu trước cleaning để có thể đối chiếu và truy vết.

> **Cảnh báo:** chạy mục này xóa dữ liệu staging cùng dimension, fact, view, materialized view và index phụ thuộc. `CONFIRM_SCHEMA_RESET = True` là xác nhận có chủ đích; sau reset phải chạy lại Mục 6–12. Không chạy mục này khi Notebook 03 đang dùng database.

Toàn bộ DDL chạy trong một transaction và rollback nếu thiếu bảng hoặc có lỗi.

**Đầu ra của mục:** năm staging tables rỗng, sẵn sàng nhận CSV. Chưa có dữ liệu để bàn giao cho Notebook 03 ở thời điểm này.

In [6]:
# Chuẩn bị DDL và yêu cầu người chạy xác nhận rõ thao tác reset có tính phá hủy dữ liệu.
CREATE_TABLES_SQL_PATH = SQL_DIR / '01_create_tables.sql'
CONFIRM_SCHEMA_RESET = False  # Đặt True nếu muốn xóa và tạo lại các staging tables

# Sinh danh sách staging table từ DATASET_CONFIG để tránh khai báo trùng lặp.
EXPECTED_STAGING_TABLES = tuple(
    config['staging_table'] for config in DATASET_CONFIG.values()
)

# Kiểm tra script tồn tại và có nội dung trước khi mở transaction database.
if not CREATE_TABLES_SQL_PATH.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy SQL script: {CREATE_TABLES_SQL_PATH}'
    )
if CREATE_TABLES_SQL_PATH.stat().st_size == 0:
    raise ValueError(f'SQL script đang rỗng: {CREATE_TABLES_SQL_PATH}')

print(f'SQL script: {CREATE_TABLES_SQL_PATH.name}')
print('Staging tables sẽ được tạo lại:')
for table_name in EXPECTED_STAGING_TABLES:
    print('-', table_name)
print(f'Xác nhận reset hiện tại: {CONFIRM_SCHEMA_RESET}')

SQL script: 01_create_tables.sql
Staging tables sẽ được tạo lại:
- staging_global
- staging_country
- staging_state
- staging_city
- staging_major_city
Xác nhận reset hiện tại: True


In [7]:
# Safety gate: cell không được xóa/tạo lại schema nếu chưa xác nhận ở cell trước.
if CONFIRM_SCHEMA_RESET is not True:
    raise PermissionError(
        'Schema reset chưa được xác nhận. Đổi CONFIRM_SCHEMA_RESET = True '
        'ở cell phía trên nếu bạn chủ động muốn xóa và tạo lại staging tables.'
    )

# Xác nhận script thật sự dùng chiến lược DROP ... CASCADE mà nhóm đã chọn.
ddl_sql = CREATE_TABLES_SQL_PATH.read_text(encoding='utf-8')
if 'DROP TABLE IF EXISTS' not in ddl_sql.upper():
    raise ValueError('SQL script không chứa cơ chế DROP TABLE IF EXISTS đã chọn.')

# Chạy toàn bộ DDL và validation trong một transaction để có thể rollback nguyên vẹn.
try:
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        try:
            with connection.cursor() as cursor:
                cursor.execute(ddl_sql)
                cursor.execute(
                    """
                    SELECT table_name
                    FROM information_schema.tables
                    WHERE table_schema = 'public'
                      AND table_name = ANY(%s)
                    ORDER BY table_name;
                    """,
                    (list(EXPECTED_STAGING_TABLES),),
                )
                # Đối chiếu catalog PostgreSQL thay vì chỉ tin rằng cursor.execute đã thành công.
                created_tables = {row[0] for row in cursor.fetchall()}
                missing_tables = set(EXPECTED_STAGING_TABLES) - created_tables
                if missing_tables:
                    missing_table_names = ', '.join(sorted(missing_tables))
                    raise RuntimeError(
                        f'Thiếu staging tables sau khi chạy DDL: {missing_table_names}'
                    )

                # Sau reset, cả 5 staging table phải tồn tại và có row count bằng 0.
                table_row_counts = []
                for table_name in EXPECTED_STAGING_TABLES:
                    cursor.execute(
                        sql.SQL('SELECT COUNT(*) FROM {}').format(
                            sql.Identifier(table_name)
                        )
                    )
                    table_row_counts.append({
                        'table': table_name,
                        'row_count': cursor.fetchone()[0],
                    })
            connection.commit()
        except Exception:
            connection.rollback()
            raise
except psycopg2.Error as error:
    raise RuntimeError(
        'Không thể tạo staging tables. Database đã rollback transaction.'
    ) from error

schema_validation_df = pd.DataFrame(table_row_counts)
display(schema_validation_df)
print('Đã tạo lại và xác nhận đủ 5 staging tables.')

,table,row_count
0,staging_global,0
1,staging_country,0
2,staging_state,0
3,staging_city,0
4,staging_major_city,0


Đã tạo lại và xác nhận đủ 5 staging tables.


## 6. Import năm CSV vào staging

Mục này thực hiện bước Load: truyền từng CSV từ máy chạy notebook vào staging bằng `COPY FROM STDIN`. Dữ liệu được stream nên PostgreSQL không cần quyền đọc đường dẫn file và Python không phải giữ hơn 5 triệu dòng City trong RAM.

Trước mỗi file, bảng đích được `TRUNCATE ... RESTART IDENTITY`. `TRUNCATE` và `COPY` nằm trong cùng transaction của dataset: thành công thì commit, lỗi thì rollback. Vì vậy có thể chạy lại toàn bộ Mục 6 mà không cộng dồn hoặc tạo bản ghi trùng do lần import trước.

`staging_id` và `loaded_at` do PostgreSQL tạo; ô CSV rỗng được nạp thành SQL `NULL`, không tự điền hoặc loại bỏ. `SQL/02_import_data.sql` tài liệu hóa mapping cột, còn notebook cung cấp dữ liệu STDIN.

**Đầu ra của mục:** `IMPORT_RESULTS` và năm staging tables chứa bản sao có kiểm soát của CSV. Đây vẫn là dữ liệu thô; Notebook 03 chưa nên cleaning trực tiếp trước khi Mục 7–10 xác nhận pipeline.

In [ ]:
# Cấu hình ánh xạ từ cột CSV sang cột staging cho lệnh PostgreSQL COPY.
import time


# SQL reference tài liệu hóa cú pháp COPY tương ứng bên ngoài notebook.
IMPORT_SQL_REFERENCE_PATH = SQL_DIR / '02_import_data.sql'
if not IMPORT_SQL_REFERENCE_PATH.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy SQL reference: {IMPORT_SQL_REFERENCE_PATH}'
    )
if IMPORT_SQL_REFERENCE_PATH.stat().st_size == 0:
    raise ValueError(f'SQL reference đang rỗng: {IMPORT_SQL_REFERENCE_PATH}')

# Tên cột đích dùng snake_case; thứ tự phải khớp header nguồn tương ứng.
COPY_COLUMN_CONFIG = {
    'global': (
        'dt', 'land_average_temperature',
        'land_average_temperature_uncertainty', 'land_max_temperature',
        'land_max_temperature_uncertainty', 'land_min_temperature',
        'land_min_temperature_uncertainty',
        'land_and_ocean_average_temperature',
        'land_and_ocean_average_temperature_uncertainty',
    ),
    'country': (
        'dt', 'average_temperature',
        'average_temperature_uncertainty', 'country',
    ),
    'state': (
        'dt', 'average_temperature',
        'average_temperature_uncertainty', 'state', 'country',
    ),
    'city': (
        'dt', 'average_temperature',
        'average_temperature_uncertainty', 'city', 'country',
        'latitude', 'longitude',
    ),
    'major_city': (
        'dt', 'average_temperature',
        'average_temperature_uncertainty', 'city', 'country',
        'latitude', 'longitude',
    ),
}

# Bảo đảm không bỏ sót hoặc khai báo thừa dataset trong cấu hình import.
if set(COPY_COLUMN_CONFIG) != set(DATASET_CONFIG):
    raise ValueError('COPY_COLUMN_CONFIG không khớp với DATASET_CONFIG.')

print(f'COPY reference: {IMPORT_SQL_REFERENCE_PATH.name}')
print(f'Đã cấu hình mapping cho {len(COPY_COLUMN_CONFIG)} dataset.')

COPY reference: 02_import_data.sql
Đã cấu hình mapping cho 5 dataset.


In [ ]:
# Hàm nhập một CSV độc lập để mỗi bảng có transaction và thông báo lỗi riêng.
def import_csv_to_staging(
    dataset_name: str,
    csv_path: Path,
    table_name: str,
    copy_columns: tuple[str, ...],
) -> dict:
    """Replace one staging table with data streamed from a local CSV."""
    if not csv_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy CSV: {csv_path}')

    started_at = time.perf_counter()
    connection = None
    try:
        connection = psycopg2.connect(**DB_CONFIG)
        connection.set_client_encoding('UTF8')
        with connection.cursor() as cursor:
            # Xóa dữ liệu lần chạy trước và đưa staging_id về 1 để lần chạy có thể tái lập.
            cursor.execute(
                sql.SQL('TRUNCATE TABLE {} RESTART IDENTITY').format(
                    sql.Identifier(table_name)
                )
            )

            # Dùng Identifier để quote an toàn tên bảng/cột; dữ liệu được stream qua STDIN.
            copy_statement = sql.SQL(
                """
                COPY {} ({})
                FROM STDIN
                WITH (
                    FORMAT CSV, HEADER TRUE, DELIMITER ',',
                    QUOTE '"', ESCAPE '"', NULL ''
                )
                """
            ).format(
                sql.Identifier(table_name),
                sql.SQL(', ').join(
                    sql.Identifier(column) for column in copy_columns
                ),
            )

            # Streaming COPY tránh nạp file City hơn 5 triệu dòng vào bộ nhớ Python.
            with csv_path.open('r', encoding='utf-8-sig', newline='') as csv_file:
                cursor.copy_expert(
                    copy_statement.as_string(connection),
                    csv_file,
                    size=1024 * 1024,
                )

            # Đếm ngay trong cùng transaction để chỉ commit khi import hoàn tất.
            cursor.execute(
                sql.SQL('SELECT COUNT(*) FROM {}').format(
                    sql.Identifier(table_name)
                )
            )
            loaded_rows = cursor.fetchone()[0]
        connection.commit()
    # Mọi lỗi đều rollback bảng hiện tại, không để lại dữ liệu import dở dang.
    except Exception as error:
        if connection is not None:
            connection.rollback()
        raise RuntimeError(
            f'Import thất bại: {csv_path.name} → {table_name}. '
            'Transaction của bảng đã rollback.'
        ) from error
    finally:
        if connection is not None:
            connection.close()

    elapsed_seconds = time.perf_counter() - started_at
    return {
        'dataset': dataset_name,
        'file': csv_path.name,
        'table': table_name,
        'loaded_rows': loaded_rows,
        'expected_rows': SOURCE_ROW_COUNTS[dataset_name],
        'row_count_match': loaded_rows == SOURCE_ROW_COUNTS[dataset_name],
        'elapsed_seconds': elapsed_seconds,
    }

In [ ]:
# Kiểm tra đủ staging tables trước khi bắt đầu tác vụ COPY tốn thời gian.
with closing(psycopg2.connect(**DB_CONFIG)) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
              AND table_name = ANY(%s);
            """,
            (list(EXPECTED_STAGING_TABLES),),
        )
        available_staging_tables = {row[0] for row in cursor.fetchall()}

missing_staging_tables = (
    set(EXPECTED_STAGING_TABLES) - available_staging_tables
)
if missing_staging_tables:
    missing_names = ', '.join(sorted(missing_staging_tables))
    raise RuntimeError(
        f'Chưa có đủ staging tables: {missing_names}. Hãy chạy lại Mục 5.'
    )

# Import tuần tự để dễ theo dõi thời gian và xác định chính xác dataset lỗi.
import_results = []
total_started_at = time.perf_counter()

for dataset_name, config in DATASET_CONFIG.items():
    csv_path = RAW_DATA_DIR / config['filename']
    table_name = config['staging_table']
    print(f'Đang import {csv_path.name} → {table_name} ...')
    result = import_csv_to_staging(
        dataset_name=dataset_name,
        csv_path=csv_path,
        table_name=table_name,
        copy_columns=COPY_COLUMN_CONFIG[dataset_name],
    )
    import_results.append(result)
    print(
        f"Hoàn thành {table_name}: {result['loaded_rows']:,} dòng "
        f"trong {result['elapsed_seconds']:,.2f} giây"
    )

# Giữ kết quả ở biến cấp notebook để Mục 7 đối chiếu với database.
IMPORT_RESULTS = pd.DataFrame(import_results)
display(IMPORT_RESULTS)

total_elapsed = time.perf_counter() - total_started_at
print(f'Tổng số dòng đã import: {IMPORT_RESULTS["loaded_rows"].sum():,}')
print(f'Tổng thời gian import: {total_elapsed:,.2f} giây')

Đang import GlobalTemperatures.csv → staging_global ...
Hoàn thành staging_global: 3,192 dòng trong 0.06 giây
Đang import GlobalLandTemperaturesByCountry.csv → staging_country ...
Hoàn thành staging_country: 577,462 dòng trong 2.38 giây
Đang import GlobalLandTemperaturesByState.csv → staging_state ...
Hoàn thành staging_state: 645,675 dòng trong 2.61 giây
Đang import GlobalLandTemperaturesByCity.csv → staging_city ...
Hoàn thành staging_city: 5,010,113 dòng trong 35.28 giây
Đang import GlobalLandTemperaturesByMajorCity.csv → staging_major_city ...
Hoàn thành staging_major_city: 239,177 dòng trong 1.20 giây


,dataset,file,table,loaded_rows,expected_rows,row_count_match,elapsed_seconds
0,global,GlobalTemperatures.csv,staging_global,3192,3192,True,0.056450
1,country,GlobalLandTemperaturesByCountry.csv,staging_country,577462,577462,True,2.380554
2,state,GlobalLandTemperaturesByState.csv,staging_state,645675,645675,True,2.608199
3,city,GlobalLandTemperaturesByCity.csv,staging_city,5010113,5010113,True,35.282339
4,major_city,GlobalLandTemperaturesByMajorCity.csv,staging_major_city,239177,239177,True,1.195999


Tổng số dòng đã import: 6,475,619
Tổng thời gian import: 41.53 giây


## 7. Đối chiếu row count và metadata staging

Mục này trả lời câu hỏi: dữ liệu PostgreSQL có đầy đủ như CSV mà Notebook 01 đã khảo sát hay không? Pipeline so sánh ba nguồn số liệu: `SOURCE_ROW_COUNTS` từ Notebook 01, `loaded_rows` của Mục 6 và `COUNT(*)` thực tế trong staging.

Ngoài row count, validation kiểm tra dải `staging_id` liên tục từ 1 và không thiếu `loaded_at`. Các truy vấn chỉ đọc; bất kỳ trạng thái `FAIL` nào đều dừng pipeline trước khi chuẩn hóa để tránh truyền lỗi sang Notebook 03.

**Đầu ra của mục:** `ROW_COUNT_VALIDATION` và `DB_ROW_COUNTS`. Khi toàn bộ `PASS`, người làm Notebook 01 có bằng chứng rằng import không cắt dòng; người làm Notebook 03 có thể dùng staging làm mốc đối chiếu lineage.

In [ ]:
# Kiểm tra dependency của Mục 7 trước khi truy vấn lại staging tables.
if 'IMPORT_RESULTS' not in globals():
    raise RuntimeError('Chưa có IMPORT_RESULTS. Hãy chạy Mục 6 trước Mục 7.')
if IMPORT_RESULTS.empty:
    raise RuntimeError('IMPORT_RESULTS đang rỗng. Hãy chạy lại Mục 6.')

required_import_columns = {'dataset', 'table', 'loaded_rows'}
missing_import_columns = required_import_columns - set(IMPORT_RESULTS.columns)
if missing_import_columns:
    missing_names = ', '.join(sorted(missing_import_columns))
    raise ValueError(f'IMPORT_RESULTS thiếu các cột: {missing_names}')

# Chuyển kết quả import thành lookup dataset → row count để đối chiếu ba phía.
imported_row_counts = {
    row.dataset: int(row.loaded_rows)
    for row in IMPORT_RESULTS.itertuples(index=False)
}

# Kết nối readonly: thu thập row count, dải ID và tính đầy đủ của loaded_at.
database_metrics = []
with closing(psycopg2.connect(**DB_CONFIG)) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        for dataset_name, config in DATASET_CONFIG.items():
            table_name = config['staging_table']
            cursor.execute(
                sql.SQL(
                    """
                    SELECT
                        COUNT(*)::BIGINT AS row_count,
                        MIN(staging_id)::BIGINT AS min_staging_id,
                        MAX(staging_id)::BIGINT AS max_staging_id,
                        COUNT(*) FILTER (WHERE loaded_at IS NULL)::BIGINT
                            AS null_loaded_at
                    FROM {}
                    """
                ).format(sql.Identifier(table_name))
            )
            row_count, min_id, max_id, null_loaded_at = cursor.fetchone()
            database_metrics.append({
                'dataset': dataset_name,
                'table': table_name,
                'database_rows': int(row_count),
                'min_staging_id': min_id,
                'max_staging_id': max_id,
                'null_loaded_at': int(null_loaded_at),
            })

database_metrics_df = pd.DataFrame(database_metrics)

In [ ]:
# Đối chiếu nguồn Notebook 01 → kết quả COPY → dữ liệu thực tế trong PostgreSQL.
validation_rows = []

for metric in database_metrics:
    dataset_name = metric['dataset']
    expected_rows = int(SOURCE_ROW_COUNTS[dataset_name])
    imported_rows = imported_row_counts.get(dataset_name)
    database_rows = metric['database_rows']
    # Sau TRUNCATE RESTART IDENTITY, ID phải liên tục từ 1 và loaded_at không được NULL.
    metadata_ok = (
        metric['min_staging_id'] == 1
        and metric['max_staging_id'] == database_rows
        and metric['null_loaded_at'] == 0
    )
    row_count_ok = (
        imported_rows is not None
        and expected_rows == imported_rows == database_rows
    )

    validation_rows.append({
        **metric,
        'expected_rows': expected_rows,
        'imported_rows': imported_rows,
        'difference_from_source': database_rows - expected_rows,
        'difference_from_import': (
            database_rows - imported_rows if imported_rows is not None else None
        ),
        'metadata_ok': metadata_ok,
        'status': 'PASS' if row_count_ok and metadata_ok else 'FAIL',
    })

ROW_COUNT_VALIDATION = pd.DataFrame(validation_rows)
DB_ROW_COUNTS = {
    row['dataset']: row['database_rows'] for row in validation_rows
}

display_columns = [
    'dataset', 'table', 'expected_rows', 'imported_rows', 'database_rows',
    'difference_from_source', 'difference_from_import',
    'min_staging_id', 'max_staging_id', 'null_loaded_at',
    'metadata_ok', 'status',
]
display(ROW_COUNT_VALIDATION[display_columns])

total_expected_rows = sum(SOURCE_ROW_COUNTS.values())
total_database_rows = sum(DB_ROW_COUNTS.values())
# Không cho phép analytical layer được tạo từ staging chưa vượt qua validation.
failed_validation = ROW_COUNT_VALIDATION.loc[
    ROW_COUNT_VALIDATION['status'] != 'PASS'
]

if not failed_validation.empty:
    failed_tables = ', '.join(failed_validation['table'].tolist())
    raise RuntimeError(
        f'Row-count validation thất bại tại: {failed_tables}. '
        'Không tiếp tục sang Mục 8.'
    )

print('Đối chiếu row count và metadata staging: PASS')
print(f'Tổng số dòng dự kiến: {total_expected_rows:,}')
print(f'Tổng số dòng PostgreSQL: {total_database_rows:,}')

,dataset,table,expected_rows,imported_rows,database_rows,difference_from_source,difference_from_import,min_staging_id,max_staging_id,null_loaded_at,metadata_ok,status
0,global,staging_global,3192,3192,3192,0,0,1,3192,0,True,PASS
1,country,staging_country,577462,577462,577462,0,0,1,577462,0,True,PASS
2,state,staging_state,645675,645675,645675,0,0,1,645675,0,True,PASS
3,city,staging_city,5010113,5010113,5010113,0,0,1,5010113,0,True,PASS
4,major_city,staging_major_city,239177,239177,239177,0,0,1,239177,0,True,PASS


Đối chiếu row count và metadata staging: PASS
Tổng số dòng dự kiến: 6,475,619
Tổng số dòng PostgreSQL: 6,475,619


## 8. Chuẩn hóa và nạp dimension/fact

Mục này thực hiện bước Transform của pipeline: chuyển staging thành mô hình dimension/fact mà không sửa dữ liệu staging. Bốn dimensions chuẩn hóa ngày và geography; năm fact tables tiếp tục giữ riêng grain Global, Country, State, City và Major City.

Transformation chỉ xử lý cấu trúc: `BTRIM` tên địa lý, chuyển tọa độ N/S/E/W thành số có dấu và tạo thuộc tính năm/tháng/quý/thập kỷ. Temperature và uncertainty `NULL` được giữ nguyên; giá trị cực đoan cũng không bị loại. Đây là ranh giới quan trọng: chuẩn hóa để join được không đồng nghĩa với làm sạch dữ liệu.

Analytical schema và `INSERT ... SELECT` chạy trong một transaction. Trước commit, row count của mỗi fact phải bằng staging tương ứng; nếu sai, toàn bộ rebuild rollback. `source_staging_id` tạo lineage từ fact về dòng staging.

**Đầu ra của mục:** `FACT_LOAD_VALIDATION`, bốn dimensions và năm fact tables. Notebook 03 có thể dùng lineage để so sánh trước/sau cleaning và phải tự tài liệu hóa mọi dòng bị sửa hoặc loại.

In [ ]:
# Analytical layer chỉ được xây sau khi toàn bộ staging validation đã PASS.
if 'ROW_COUNT_VALIDATION' not in globals():
    raise RuntimeError('Chưa có ROW_COUNT_VALIDATION. Hãy chạy Mục 7 trước.')
if (ROW_COUNT_VALIDATION['status'] != 'PASS').any():
    raise RuntimeError('Mục 7 chưa PASS toàn bộ; không thể xây analytical layer.')

# Chỉ trích phần analytical DDL; staging vừa import phải được giữ nguyên.
ANALYTICAL_START_MARKER = '-- BEGIN ANALYTICAL_SCHEMA'
ANALYTICAL_END_MARKER = '-- END ANALYTICAL_SCHEMA'
schema_sql_text = CREATE_TABLES_SQL_PATH.read_text(encoding='utf-8')

if (
    ANALYTICAL_START_MARKER not in schema_sql_text
    or ANALYTICAL_END_MARKER not in schema_sql_text
):
    raise ValueError('Không tìm thấy analytical schema block trong 01_create_tables.sql.')

analytical_section = schema_sql_text.split(ANALYTICAL_START_MARKER, 1)[1]
ANALYTICAL_DDL_SQL = analytical_section.split(
    ANALYTICAL_END_MARKER, 1
)[0].strip()

# Danh sách này được dùng để nạp và kiểm tra row count sau transformation.
DIMENSION_TABLES = ('dim_date', 'dim_country', 'dim_state', 'dim_city')
FACT_TABLE_CONFIG = {
    'global': 'fact_global_temperature',
    'country': 'fact_country_temperature',
    'state': 'fact_state_temperature',
    'city': 'fact_city_temperature',
    'major_city': 'fact_major_city_temperature',
}
ANALYTICAL_TABLES = DIMENSION_TABLES + tuple(FACT_TABLE_CONFIG.values())

print(f'Dimensions: {len(DIMENSION_TABLES)}')
print(f'Fact tables: {len(FACT_TABLE_CONFIG)}')
print('Staging tables sẽ được giữ nguyên.')

Dimensions: 4
Fact tables: 5
Staging tables sẽ được giữ nguyên.


In [ ]:
# Chuyển tọa độ dạng chuỗi (N/S/E/W) thành số có dấu ngay trong PostgreSQL.
def directional_coordinate_sql(
    column_expression: str, positive_direction: str, negative_direction: str
) -> str:
    """Build a PostgreSQL expression that converts 34.56N/23.31S to signed numeric values."""
    directions = positive_direction + negative_direction
    return f"""
        CASE
            WHEN BTRIM({column_expression})
                 ~ '^[0-9]+([.][0-9]+)?[{directions}]$'
            THEN LEFT(BTRIM({column_expression}), -1)::DOUBLE PRECISION
                 * CASE RIGHT(BTRIM({column_expression}), 1)
                       WHEN '{negative_direction}' THEN -1.0
                       ELSE 1.0
                   END
            ELSE NULL
        END
    """.strip()


# Hai biểu thức được tái sử dụng khi tạo dimension và nạp hai city facts.
LATITUDE_SQL = directional_coordinate_sql('s.latitude', 'N', 'S')
LONGITUDE_SQL = directional_coordinate_sql('s.longitude', 'E', 'W')

# Dictionary giữ đúng thứ tự nạp: dimensions trước, facts sau để thỏa foreign keys.
TRANSFORMATION_SQL = {
    # Một ngày duy nhất được hợp nhất từ cả 5 nguồn.
    'load_dim_date': """
        INSERT INTO dim_date (full_date, year, month, quarter, decade)
        SELECT
            full_date,
            EXTRACT(YEAR FROM full_date)::SMALLINT,
            EXTRACT(MONTH FROM full_date)::SMALLINT,
            EXTRACT(QUARTER FROM full_date)::SMALLINT,
            ((EXTRACT(YEAR FROM full_date)::INTEGER / 10) * 10)::SMALLINT
        FROM (
            SELECT dt AS full_date FROM staging_global WHERE dt IS NOT NULL
            UNION
            SELECT dt FROM staging_country WHERE dt IS NOT NULL
            UNION
            SELECT dt FROM staging_state WHERE dt IS NOT NULL
            UNION
            SELECT dt FROM staging_city WHERE dt IS NOT NULL
            UNION
            SELECT dt FROM staging_major_city WHERE dt IS NOT NULL
        ) AS source_dates
        ORDER BY full_date;
    """,
    # BTRIM/NULLIF chuẩn hóa khóa join nhưng chưa thay đổi giá trị nhiệt độ.
    'load_dim_country': """
        INSERT INTO dim_country (country_name)
        SELECT country_name
        FROM (
            SELECT NULLIF(BTRIM(country), '') AS country_name FROM staging_country
            UNION
            SELECT NULLIF(BTRIM(country), '') FROM staging_state
            UNION
            SELECT NULLIF(BTRIM(country), '') FROM staging_city
            UNION
            SELECT NULLIF(BTRIM(country), '') FROM staging_major_city
        ) AS source_countries
        WHERE country_name IS NOT NULL
        ORDER BY country_name;
    """,
    'load_dim_state': """
        INSERT INTO dim_state (state_name, country_id)
        SELECT DISTINCT BTRIM(s.state), c.country_id
        FROM staging_state AS s
        JOIN dim_country AS c ON c.country_name = BTRIM(s.country)
        WHERE NULLIF(BTRIM(s.state), '') IS NOT NULL
        ORDER BY BTRIM(s.state), c.country_id;
    """,
    # Hợp nhất City và Major City; BOOL_OR giữ cờ major nếu xuất hiện ở ít nhất một nguồn.
    'load_dim_city': f"""
        WITH city_source AS (
            SELECT
                NULLIF(BTRIM(s.city), '') AS city_name,
                NULLIF(BTRIM(s.country), '') AS country_name,
                {LATITUDE_SQL} AS latitude,
                {LONGITUDE_SQL} AS longitude,
                FALSE AS is_major_city
            FROM staging_city AS s
            UNION ALL
            SELECT
                NULLIF(BTRIM(s.city), '') AS city_name,
                NULLIF(BTRIM(s.country), '') AS country_name,
                {LATITUDE_SQL} AS latitude,
                {LONGITUDE_SQL} AS longitude,
                TRUE AS is_major_city
            FROM staging_major_city AS s
        )
        INSERT INTO dim_city (
            city_name, country_id, latitude, longitude, is_major_city
        )
        SELECT
            cs.city_name, c.country_id, cs.latitude, cs.longitude,
            BOOL_OR(cs.is_major_city)
        FROM city_source AS cs
        JOIN dim_country AS c ON c.country_name = cs.country_name
        WHERE cs.city_name IS NOT NULL
          AND cs.country_name IS NOT NULL
          AND cs.latitude IS NOT NULL
          AND cs.longitude IS NOT NULL
        GROUP BY cs.city_name, c.country_id, cs.latitude, cs.longitude
        ORDER BY cs.city_name, c.country_id, cs.latitude, cs.longitude;
    """,
    # Các fact giữ source_staging_id để Notebook 03 truy vết về dòng nguồn.
    'load_fact_global': """
        INSERT INTO fact_global_temperature (
            date_id, source_staging_id, land_average_temperature,
            land_average_temperature_uncertainty, land_max_temperature,
            land_max_temperature_uncertainty, land_min_temperature,
            land_min_temperature_uncertainty,
            land_and_ocean_average_temperature,
            land_and_ocean_average_temperature_uncertainty
        )
        SELECT
            d.date_id, s.staging_id, s.land_average_temperature,
            s.land_average_temperature_uncertainty, s.land_max_temperature,
            s.land_max_temperature_uncertainty, s.land_min_temperature,
            s.land_min_temperature_uncertainty,
            s.land_and_ocean_average_temperature,
            s.land_and_ocean_average_temperature_uncertainty
        FROM staging_global AS s
        JOIN dim_date AS d ON d.full_date = s.dt;
    """,
    'load_fact_country': """
        INSERT INTO fact_country_temperature (
            date_id, country_id, source_staging_id,
            average_temperature, average_temperature_uncertainty
        )
        SELECT
            d.date_id, c.country_id, s.staging_id,
            s.average_temperature, s.average_temperature_uncertainty
        FROM staging_country AS s
        JOIN dim_date AS d ON d.full_date = s.dt
        JOIN dim_country AS c ON c.country_name = BTRIM(s.country);
    """,
    'load_fact_state': """
        INSERT INTO fact_state_temperature (
            date_id, state_id, source_staging_id,
            average_temperature, average_temperature_uncertainty
        )
        SELECT
            d.date_id, st.state_id, s.staging_id,
            s.average_temperature, s.average_temperature_uncertainty
        FROM staging_state AS s
        JOIN dim_date AS d ON d.full_date = s.dt
        JOIN dim_country AS c ON c.country_name = BTRIM(s.country)
        JOIN dim_state AS st
          ON st.country_id = c.country_id
         AND st.state_name = BTRIM(s.state);
    """,
    'load_fact_city': f"""
        WITH normalized_city AS (
            SELECT
                s.staging_id, s.dt, s.average_temperature,
                s.average_temperature_uncertainty,
                BTRIM(s.city) AS city_name,
                BTRIM(s.country) AS country_name,
                {LATITUDE_SQL} AS latitude,
                {LONGITUDE_SQL} AS longitude
            FROM staging_city AS s
        )
        INSERT INTO fact_city_temperature (
            date_id, city_id, source_staging_id,
            average_temperature, average_temperature_uncertainty
        )
        SELECT
            d.date_id, ci.city_id, n.staging_id,
            n.average_temperature, n.average_temperature_uncertainty
        FROM normalized_city AS n
        JOIN dim_date AS d ON d.full_date = n.dt
        JOIN dim_country AS c ON c.country_name = n.country_name
        JOIN dim_city AS ci
          ON ci.country_id = c.country_id
         AND ci.city_name = n.city_name
         AND ci.latitude = n.latitude
         AND ci.longitude = n.longitude;
    """,
    'load_fact_major_city': f"""
        WITH normalized_city AS (
            SELECT
                s.staging_id, s.dt, s.average_temperature,
                s.average_temperature_uncertainty,
                BTRIM(s.city) AS city_name,
                BTRIM(s.country) AS country_name,
                {LATITUDE_SQL} AS latitude,
                {LONGITUDE_SQL} AS longitude
            FROM staging_major_city AS s
        )
        INSERT INTO fact_major_city_temperature (
            date_id, city_id, source_staging_id,
            average_temperature, average_temperature_uncertainty
        )
        SELECT
            d.date_id, ci.city_id, n.staging_id,
            n.average_temperature, n.average_temperature_uncertainty
        FROM normalized_city AS n
        JOIN dim_date AS d ON d.full_date = n.dt
        JOIN dim_country AS c ON c.country_name = n.country_name
        JOIN dim_city AS ci
          ON ci.country_id = c.country_id
         AND ci.city_name = n.city_name
         AND ci.latitude = n.latitude
         AND ci.longitude = n.longitude;
    """,
}

print(f'Đã chuẩn bị {len(TRANSFORMATION_SQL)} bước transformation.')

Đã chuẩn bị 9 bước transformation.


In [ ]:
# Rebuild toàn bộ analytical layer trong một transaction all-or-nothing.
analytical_started_at = time.perf_counter()
current_step = 'connect_database'
connection = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    with connection.cursor() as cursor:
        current_step = 'recreate_analytical_schema'
        cursor.execute(ANALYTICAL_DDL_SQL)

        # Chạy transformation theo thứ tự dictionary và ghi số dòng/thời gian từng bước.
        transformation_results = []
        for step_name, statement in TRANSFORMATION_SQL.items():
            current_step = step_name
            print(f'Đang chạy {step_name} ...')
            step_started_at = time.perf_counter()
            cursor.execute(statement)
            transformation_results.append({
                'step': step_name,
                'inserted_rows': cursor.rowcount,
                'elapsed_seconds': time.perf_counter() - step_started_at,
            })

        # Đếm tất cả dimensions/facts trước khi commit để phát hiện nạp thiếu.
        current_step = 'validate_analytical_row_counts'
        analytical_counts = []
        for table_name in ANALYTICAL_TABLES:
            cursor.execute(
                sql.SQL('SELECT COUNT(*) FROM {}').format(
                    sql.Identifier(table_name)
                )
            )
            analytical_counts.append({
                'table': table_name,
                'row_count': int(cursor.fetchone()[0]),
            })

        fact_count_by_table = {
            row['table']: row['row_count']
            for row in analytical_counts
            if row['table'] in FACT_TABLE_CONFIG.values()
        }
        # Mỗi fact phải bảo toàn đúng row count của staging tương ứng.
        fact_load_validation = []
        for dataset_name, fact_table in FACT_TABLE_CONFIG.items():
            expected_rows = int(SOURCE_ROW_COUNTS[dataset_name])
            fact_rows = fact_count_by_table.get(fact_table, 0)
            fact_load_validation.append({
                'dataset': dataset_name,
                'fact_table': fact_table,
                'expected_rows': expected_rows,
                'fact_rows': fact_rows,
                'difference': fact_rows - expected_rows,
                'status': 'PASS' if fact_rows == expected_rows else 'FAIL',
            })

        failed_fact_loads = [
            row for row in fact_load_validation if row['status'] != 'PASS'
        ]
        if failed_fact_loads:
            failed_names = ', '.join(
                row['fact_table'] for row in failed_fact_loads
            )
            raise RuntimeError(
                f'Fact row count không khớp staging: {failed_names}'
            )

    # Chỉ commit khi schema, transformations và fact validation đều thành công.
    connection.commit()
except Exception as error:
    if connection is not None:
        connection.rollback()
    raise RuntimeError(
        f'Analytical rebuild thất bại tại bước {current_step}. '
        'Transaction đã rollback.'
    ) from error
finally:
    if connection is not None:
        connection.close()

# Xuất các bảng kết quả để cell tiếp theo trình bày và Mục 9 sử dụng.
TRANSFORMATION_RESULTS = pd.DataFrame(transformation_results)
ANALYTICAL_ROW_COUNTS = pd.DataFrame(analytical_counts)
FACT_LOAD_VALIDATION = pd.DataFrame(fact_load_validation)
analytical_elapsed = time.perf_counter() - analytical_started_at

Đang chạy load_dim_date ...
Đang chạy load_dim_country ...
Đang chạy load_dim_state ...
Đang chạy load_dim_city ...
Đang chạy load_fact_global ...
Đang chạy load_fact_country ...
Đang chạy load_fact_state ...
Đang chạy load_fact_city ...
Đang chạy load_fact_major_city ...


In [ ]:
# Trình bày riêng hiệu năng, row count và validation để người đọc dễ kiểm tra.
print('Kết quả các bước transformation:')
display(TRANSFORMATION_RESULTS)

print('Row count analytical tables:')
display(ANALYTICAL_ROW_COUNTS)

print('Đối chiếu fact với staging:')
display(FACT_LOAD_VALIDATION)

print('Analytical layer đã được tạo và nạp thành công.')
print(f'Tổng thời gian: {analytical_elapsed:,.2f} giây')

Kết quả các bước transformation:


,step,inserted_rows,elapsed_seconds
0,load_dim_date,3266,2.134109
1,load_dim_country,243,11.344417
2,load_dim_state,241,1.586246
3,load_dim_city,2795,60.739509
4,load_fact_global,3192,0.063426
5,load_fact_country,577462,17.433770
6,load_fact_state,645675,20.218327
7,load_fact_city,5010113,251.037714
8,load_fact_major_city,239177,11.286120


Row count analytical tables:


,table,row_count
0,dim_date,3266
1,dim_country,243
2,dim_state,241
3,dim_city,2795
4,fact_global_temperature,3192
5,fact_country_temperature,577462
6,fact_state_temperature,645675
7,fact_city_temperature,5010113
8,fact_major_city_temperature,239177


Đối chiếu fact với staging:


,dataset,fact_table,expected_rows,fact_rows,difference,status
0,global,fact_global_temperature,3192,3192,0,PASS
1,country,fact_country_temperature,577462,577462,0,PASS
2,state,fact_state_temperature,645675,645675,0,PASS
3,city,fact_city_temperature,5010113,5010113,0,PASS
4,major_city,fact_major_city_temperature,239177,239177,0,PASS


Analytical layer đã được tạo và nạp thành công.
Tổng thời gian: 376.43 giây


## 9. Tạo analytical views

Mục này tạo giao diện đọc dữ liệu chi tiết theo tháng. Mỗi `vw_*_temperature` join một fact table với dimensions để người dùng thấy ngày, năm/tháng/quý/thập kỷ, tên geography, tọa độ và các giá trị nhiệt độ mà không phải tự viết lại join kỹ thuật.

Views không lưu bản sao, không làm tròn, không loại temperature `NULL` và vẫn chứa `source_staging_id`. Script `SQL/03_views.sql` chạy trong transaction; notebook kiểm tra sự tồn tại, đúng thứ tự cột và lấy mẫu trước khi commit. Join giữa nhiều cấp độ được kiểm tra riêng ở Mục 10.

**Đầu ra của mục:** `VIEW_VALIDATION`, `VIEW_SAMPLES` và năm monthly views. Đây là nguồn đầu vào ưu tiên cho Notebook 03 vì còn giữ grain theo tháng và lineage; yearly materialized views ở Mục 11 không thay thế nguồn chi tiết này.

In [ ]:
# Chỉ công bố views khi analytical facts đã được nạp đầy đủ và validation PASS.
if 'FACT_LOAD_VALIDATION' not in globals():
    raise RuntimeError('Chưa có FACT_LOAD_VALIDATION. Hãy chạy Mục 8 trước.')
if (FACT_LOAD_VALIDATION['status'] != 'PASS').any():
    raise RuntimeError('Fact load chưa PASS toàn bộ; không thể tạo views.')

# Đọc view definitions từ SQL version-controlled thay vì viết DDL trực tiếp trong cell.
VIEWS_SQL_PATH = SQL_DIR / '03_views.sql'
if not VIEWS_SQL_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy SQL script: {VIEWS_SQL_PATH}')
if VIEWS_SQL_PATH.stat().st_size == 0:
    raise ValueError(f'SQL script đang rỗng: {VIEWS_SQL_PATH}')

# Data contract ghi rõ tên và thứ tự cột mà Notebook 03 sẽ nhận.
EXPECTED_VIEW_COLUMNS = {
    'vw_global_temperature': (
        'global_temperature_id', 'source_staging_id', 'date_id',
        'observation_date', 'year', 'month', 'quarter', 'decade',
        'land_average_temperature',
        'land_average_temperature_uncertainty', 'land_max_temperature',
        'land_max_temperature_uncertainty', 'land_min_temperature',
        'land_min_temperature_uncertainty',
        'land_and_ocean_average_temperature',
        'land_and_ocean_average_temperature_uncertainty',
    ),
    'vw_country_temperature': (
        'country_temperature_id', 'source_staging_id', 'date_id',
        'observation_date', 'year', 'month', 'quarter', 'decade',
        'country_id', 'country_name', 'average_temperature',
        'average_temperature_uncertainty',
    ),
    'vw_state_temperature': (
        'state_temperature_id', 'source_staging_id', 'date_id',
        'observation_date', 'year', 'month', 'quarter', 'decade',
        'state_id', 'state_name', 'country_id', 'country_name',
        'average_temperature', 'average_temperature_uncertainty',
    ),
    'vw_city_temperature': (
        'city_temperature_id', 'source_staging_id', 'date_id',
        'observation_date', 'year', 'month', 'quarter', 'decade',
        'city_id', 'city_name', 'country_id', 'country_name',
        'latitude', 'longitude', 'is_major_city',
        'average_temperature', 'average_temperature_uncertainty',
    ),
    'vw_major_city_temperature': (
        'major_city_temperature_id', 'source_staging_id', 'date_id',
        'observation_date', 'year', 'month', 'quarter', 'decade',
        'city_id', 'city_name', 'country_id', 'country_name',
        'latitude', 'longitude', 'is_major_city',
        'average_temperature', 'average_temperature_uncertainty',
    ),
}
EXPECTED_VIEWS = tuple(EXPECTED_VIEW_COLUMNS)
print(f'Đã cấu hình kiểm tra {len(EXPECTED_VIEWS)} views.')

Đã cấu hình kiểm tra 5 views.


In [ ]:
# Tạo và kiểm tra 5 analytical views trong cùng một transaction.
views_sql = VIEWS_SQL_PATH.read_text(encoding='utf-8')
current_view = 'execute_03_views.sql'
connection = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    with connection.cursor() as cursor:
        # Thực thi DDL rồi đọc catalog để xác nhận từng view thực sự tồn tại.
        cursor.execute(views_sql)
        cursor.execute(
            """
            SELECT table_name
            FROM information_schema.views
            WHERE table_schema = 'public'
              AND table_name = ANY(%s);
            """,
            (list(EXPECTED_VIEWS),),
        )
        created_views = {row[0] for row in cursor.fetchall()}

        # Kiểm tra schema chính xác và chỉ lấy 3 dòng mẫu khi schema hợp lệ.
        view_validation_rows = []
        view_samples = {}
        for view_name, expected_columns in EXPECTED_VIEW_COLUMNS.items():
            current_view = view_name
            cursor.execute(
                """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'public'
                  AND table_name = %s
                ORDER BY ordinal_position;
                """,
                (view_name,),
            )
            actual_columns = tuple(row[0] for row in cursor.fetchall())
            view_exists = view_name in created_views
            columns_ok = actual_columns == expected_columns

            sample_row_count = 0
            if view_exists and columns_ok:
                cursor.execute(
                    sql.SQL('SELECT * FROM {} LIMIT 3').format(
                        sql.Identifier(view_name)
                    )
                )
                sample_rows = cursor.fetchall()
                sample_columns = [description[0] for description in cursor.description]
                view_samples[view_name] = pd.DataFrame(
                    sample_rows, columns=sample_columns
                )
                sample_row_count = len(sample_rows)

            view_validation_rows.append({
                'view': view_name,
                'exists': view_exists,
                'column_count': len(actual_columns),
                'columns_ok': columns_ok,
                'sample_rows': sample_row_count,
                'status': 'PASS' if view_exists and columns_ok else 'FAIL',
            })

        failed_views = [
            row for row in view_validation_rows if row['status'] != 'PASS'
        ]
        if failed_views:
            failed_names = ', '.join(row['view'] for row in failed_views)
            raise RuntimeError(f'View validation thất bại: {failed_names}')

    # Nếu bất kỳ view nào sai data contract, khối except sẽ rollback toàn bộ lần tạo.
    connection.commit()
except Exception as error:
    if connection is not None:
        connection.rollback()
    raise RuntimeError(
        f'Tạo views thất bại tại {current_view}. Transaction đã rollback.'
    ) from error
finally:
    if connection is not None:
        connection.close()

# Lưu validation và mẫu ở cấp notebook cho Mục 10/final audit tái sử dụng.
VIEW_VALIDATION = pd.DataFrame(view_validation_rows)
VIEW_SAMPLES = view_samples

In [ ]:
# Hiển thị schema validation trước, sau đó mới hiển thị mẫu dữ liệu từng view.
display(VIEW_VALIDATION)

for view_name, sample_df in VIEW_SAMPLES.items():
    print(f'\n{view_name} — sample')
    display(sample_df)

print('Đã tạo và xác nhận đủ 5 analytical views.')

,view,exists,column_count,columns_ok,sample_rows,status
0,vw_global_temperature,True,16,True,3,PASS
1,vw_country_temperature,True,12,True,3,PASS
2,vw_state_temperature,True,14,True,3,PASS
3,vw_city_temperature,True,17,True,3,PASS
4,vw_major_city_temperature,True,17,True,3,PASS



vw_global_temperature — sample


,global_temperature_id,source_staging_id,date_id,observation_date,year,month,quarter,decade,land_average_temperature,land_average_temperature_uncertainty,land_max_temperature,land_max_temperature_uncertainty,land_min_temperature,land_min_temperature_uncertainty,land_and_ocean_average_temperature,land_and_ocean_average_temperature_uncertainty
0,1,1,75,1750-01-01,1750,1,1,1750,3.034,3.574,None,None,None,None,None,None
1,2,2,76,1750-02-01,1750,2,1,1750,3.083,3.702,None,None,None,None,None,None
2,3,3,77,1750-03-01,1750,3,1,1750,5.626,3.076,None,None,None,None,None,None



vw_country_temperature — sample


,country_temperature_id,source_staging_id,date_id,observation_date,year,month,quarter,decade,country_id,country_name,average_temperature,average_temperature_uncertainty
0,1,1,1,1743-11-01,1743,11,4,1740,3,Åland,4.384,2.294
1,7311,7311,1,1743-11-01,1743,11,4,1740,4,Albania,8.620,2.268
2,15032,15032,1,1743-11-01,1743,11,4,1740,7,Andorra,7.556,2.188


...


vw_city_temperature — sample


,city_temperature_id,source_staging_id,date_id,observation_date,year,month,quarter,decade,city_id,city_name,country_id,country_name,latitude,longitude,is_major_city,average_temperature,average_temperature_uncertainty
0,9054,9054,1431,1863-01-01,1863,1,1,1860,1,A Coruña,208,Spain,42.59,-8.73,False,8.131,2.43
1,10863,10863,1431,1863-01-01,1863,1,1,1860,2,Aachen,87,Germany,50.63,6.34,False,2.992,2.23
2,12675,12675,1431,1863-01-01,1863,1,1,1860,3,Aalborg,60,Denmark,57.05,10.33,False,3.265,2.19



vw_major_city_temperature — sample


,major_city_temperature_id,source_staging_id,date_id,observation_date,year,month,quarter,decade,city_id,city_name,country_id,country_name,latitude,longitude,is_major_city,average_temperature,average_temperature_uncertainty
0,24501,24501,1,1743-11-01,1743,11,4,1740,275,Berlin,87,Germany,52.24,13.14,True,6.326,1.601
1,51675,51675,1,1743-11-01,1743,11,4,1740,567,Chicago,234,United States,42.59,-87.27,True,5.436,2.205
2,90256,90256,1,1743-11-01,1743,11,4,1740,1057,Istanbul,224,Turkey,40.99,29.82,True,10.365,2.325


Đã tạo và xác nhận đủ 5 analytical views.


## 10. Thực hiện joins và kiểm tra unmatched rows

Mục này kiểm tra bốn quan hệ Country–Global, State–Country, City–Country và Major City–Country bằng `LEFT JOIN`. Mục tiêu không phải tạo một bảng khổng lồ, mà chứng minh các khóa có thể liên kết mà không làm nhân dòng hoặc che giấu dữ liệu nguồn.

`expected_unmatched` là khoảng trống đã tồn tại trong CSV: ví dụ Country bắt đầu trước Global hoặc một Country–Date không có trong `staging_country`. `unexpected_unmatched` là khóa có trong staging nhưng bị mất ở fact/view và được xem là lỗi pipeline. Temperature `NULL` không phải unmatched khi khóa thời gian/geography vẫn tồn tại.

Mục chỉ đọc, có timeout và chỉ lấy mẫu khi phát hiện lỗi thật để tránh quét City nhiều lần.

**Đầu ra của mục:** `JOIN_VALIDATION`, `JOIN_UNMATCHED_SAMPLES` và mẫu City–Country–Global. Notebook 03 không nên tự động xóa `expected_unmatched`; chúng là đặc điểm coverage cần được ghi nhận khi xây quy tắc cleaning.

In [ ]:
# Runtime setup để Mục 10 chạy được cả sau khi kernel vừa khởi động lại.
import os
from contextlib import closing
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv


# Cho phép Mục 10 tự khôi phục cấu hình khi được chạy sau một kernel restart.
def find_runtime_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.env').is_file() and (candidate / 'SQL').is_dir():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy project root chứa .env và SQL. '
        'Hãy mở notebook từ thư mục dự án.'
    )


# Tái tạo DB_CONFIG chỉ khi các mục trước chưa tạo biến này.
if 'DB_CONFIG' not in globals():
    PROJECT_ROOT = find_runtime_project_root(Path.cwd().resolve())
    ENV_PATH = PROJECT_ROOT / '.env'
    load_dotenv(dotenv_path=ENV_PATH, override=False)

    required_env = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
    missing_env = [name for name in required_env if not os.getenv(name)]
    if missing_env:
        raise EnvironmentError(
            'Thiếu biến trong .env: ' + ', '.join(missing_env)
        )

    try:
        runtime_db_port = int(os.environ['DB_PORT'])
    except ValueError as error:
        raise ValueError('DB_PORT phải là số nguyên, ví dụ 5432.') from error

    DB_CONFIG = {
        'host': os.environ['DB_HOST'],
        'port': runtime_db_port,
        'dbname': os.environ['DB_NAME'],
        'user': os.environ['DB_USER'],
        'password': os.environ['DB_PASSWORD'],
        'connect_timeout': 10,
    }
    if os.getenv('DB_SSLMODE'):
        DB_CONFIG['sslmode'] = os.environ['DB_SSLMODE']

# Cả 5 monthly views là dependency tối thiểu của join validation.
REQUIRED_JOIN_VIEWS = (
    'vw_global_temperature',
    'vw_country_temperature',
    'vw_state_temperature',
    'vw_city_temperature',
    'vw_major_city_temperature',
)

# Catalog check readonly giúp báo thiếu view trước khi chạy các join lớn.
try:
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute(
                """
                SELECT table_name
                FROM information_schema.views
                WHERE table_schema = 'public'
                  AND table_name = ANY(%s);
                """,
                (list(REQUIRED_JOIN_VIEWS),),
            )
            available_join_views = {row[0] for row in cursor.fetchall()}
except psycopg2.Error as error:
    raise ConnectionError(
        'Không thể kiểm tra các analytical views trong PostgreSQL.'
    ) from error

missing_join_views = set(REQUIRED_JOIN_VIEWS) - available_join_views
if missing_join_views:
    raise RuntimeError(
        'Thiếu analytical views: ' + ', '.join(sorted(missing_join_views))
        + '. Hãy chạy Mục 9 trước.'
    )

print('Runtime và 5 analytical views đã sẵn sàng cho join validation.')

# Mỗi truy vấn đo bảo toàn số dòng và phân loại unmatched expected/unexpected.
JOIN_VALIDATION_SQL = {
    # Country ngoài khoảng thời gian Global được xem là unmatched hợp lệ từ nguồn.
    'country_to_global': """
        WITH global_range AS (
            SELECT MIN(observation_date) AS min_date,
                   MAX(observation_date) AS max_date
            FROM vw_global_temperature
        ),
        source_rows AS (
            SELECT COUNT(*)::BIGINT AS row_count
            FROM vw_country_temperature
        ),
        joined AS (
            SELECT
                c.country_temperature_id AS source_id,
                c.observation_date,
                g.global_temperature_id AS target_id,
                r.min_date,
                r.max_date
            FROM vw_country_temperature AS c
            CROSS JOIN global_range AS r
            LEFT JOIN vw_global_temperature AS g ON g.date_id = c.date_id
        )
        SELECT
            s.row_count AS source_rows,
            COUNT(*)::BIGINT AS joined_rows,
            COUNT(target_id)::BIGINT AS matched_rows,
            COUNT(*) FILTER (WHERE target_id IS NULL)::BIGINT AS unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL
                  AND observation_date NOT BETWEEN min_date AND max_date
            )::BIGINT AS expected_unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL
                  AND observation_date BETWEEN min_date AND max_date
            )::BIGINT AS unexpected_unmatched_rows
        FROM joined
        CROSS JOIN source_rows AS s
        GROUP BY s.row_count;
    """,
    # Với các cấp địa lý, staging_country phân biệt thiếu từ nguồn và mất do pipeline.
    'state_to_country': """
        WITH raw_country_keys AS (
            SELECT DISTINCT dt, country FROM staging_country
        ),
        source_rows AS (
            SELECT COUNT(*)::BIGINT AS row_count FROM vw_state_temperature
        ),
        joined AS (
            SELECT
                s.state_temperature_id AS source_id,
                c.country_temperature_id AS target_id,
                raw.country AS raw_country_key
            FROM vw_state_temperature AS s
            LEFT JOIN vw_country_temperature AS c
              ON c.date_id = s.date_id
             AND c.country_id = s.country_id
            LEFT JOIN raw_country_keys AS raw
              ON raw.dt = s.observation_date
             AND raw.country = s.country_name
        )
        SELECT
            s.row_count AS source_rows,
            COUNT(*)::BIGINT AS joined_rows,
            COUNT(target_id)::BIGINT AS matched_rows,
            COUNT(*) FILTER (WHERE target_id IS NULL)::BIGINT AS unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL AND raw_country_key IS NULL
            )::BIGINT AS expected_unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL AND raw_country_key IS NOT NULL
            )::BIGINT AS unexpected_unmatched_rows
        FROM joined
        CROSS JOIN source_rows AS s
        GROUP BY s.row_count;
    """,
    'city_to_country': """
        WITH raw_country_keys AS (
            SELECT DISTINCT dt, country FROM staging_country
        ),
        source_rows AS (
            SELECT COUNT(*)::BIGINT AS row_count FROM vw_city_temperature
        ),
        joined AS (
            SELECT
                ci.city_temperature_id AS source_id,
                c.country_temperature_id AS target_id,
                raw.country AS raw_country_key
            FROM vw_city_temperature AS ci
            LEFT JOIN vw_country_temperature AS c
              ON c.date_id = ci.date_id
             AND c.country_id = ci.country_id
            LEFT JOIN raw_country_keys AS raw
              ON raw.dt = ci.observation_date
             AND raw.country = ci.country_name
        )
        SELECT
            s.row_count AS source_rows,
            COUNT(*)::BIGINT AS joined_rows,
            COUNT(target_id)::BIGINT AS matched_rows,
            COUNT(*) FILTER (WHERE target_id IS NULL)::BIGINT AS unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL AND raw_country_key IS NULL
            )::BIGINT AS expected_unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL AND raw_country_key IS NOT NULL
            )::BIGINT AS unexpected_unmatched_rows
        FROM joined
        CROSS JOIN source_rows AS s
        GROUP BY s.row_count;
    """,
    'major_city_to_country': """
        WITH raw_country_keys AS (
            SELECT DISTINCT dt, country FROM staging_country
        ),
        source_rows AS (
            SELECT COUNT(*)::BIGINT AS row_count
            FROM vw_major_city_temperature
        ),
        joined AS (
            SELECT
                ci.major_city_temperature_id AS source_id,
                c.country_temperature_id AS target_id,
                raw.country AS raw_country_key
            FROM vw_major_city_temperature AS ci
            LEFT JOIN vw_country_temperature AS c
              ON c.date_id = ci.date_id
             AND c.country_id = ci.country_id
            LEFT JOIN raw_country_keys AS raw
              ON raw.dt = ci.observation_date
             AND raw.country = ci.country_name
        )
        SELECT
            s.row_count AS source_rows,
            COUNT(*)::BIGINT AS joined_rows,
            COUNT(target_id)::BIGINT AS matched_rows,
            COUNT(*) FILTER (WHERE target_id IS NULL)::BIGINT AS unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL AND raw_country_key IS NULL
            )::BIGINT AS expected_unmatched_rows,
            COUNT(*) FILTER (
                WHERE target_id IS NULL AND raw_country_key IS NOT NULL
            )::BIGINT AS unexpected_unmatched_rows
        FROM joined
        CROSS JOIN source_rows AS s
        GROUP BY s.row_count;
    """,
}

print(f'Đã chuẩn bị {len(JOIN_VALIDATION_SQL)} phép join validation.')

Runtime và 5 analytical views đã sẵn sàng cho join validation.
Đã chuẩn bị 4 phép join validation.


In [ ]:
# Các truy vấn chẩn đoán chỉ lấy tối đa 10 dòng nếu xuất hiện unmatched bất thường.
UNMATCHED_SAMPLE_SQL = {
    'country_to_global': """
        WITH global_range AS (
            SELECT MIN(observation_date) AS min_date,
                   MAX(observation_date) AS max_date
            FROM vw_global_temperature
        )
        SELECT
            c.observation_date, c.country_id, c.country_name,
            CASE
                WHEN c.observation_date BETWEEN r.min_date AND r.max_date
                    THEN 'missing_date_within_global_range'
                ELSE 'outside_global_date_range'
            END AS unmatched_reason
        FROM vw_country_temperature AS c
        CROSS JOIN global_range AS r
        LEFT JOIN vw_global_temperature AS g ON g.date_id = c.date_id
        WHERE g.global_temperature_id IS NULL
        ORDER BY c.observation_date, c.country_name
        LIMIT 10;
    """,
    'state_to_country': """
        WITH raw_country_keys AS (
            SELECT DISTINCT dt, country FROM staging_country
        )
        SELECT
            s.observation_date, s.state_id, s.state_name,
            s.country_id, s.country_name,
            CASE WHEN raw.country IS NULL
                THEN 'absent_from_country_source'
                ELSE 'lost_during_country_pipeline'
            END AS unmatched_reason
        FROM vw_state_temperature AS s
        LEFT JOIN vw_country_temperature AS c
          ON c.date_id = s.date_id AND c.country_id = s.country_id
        LEFT JOIN raw_country_keys AS raw
          ON raw.dt = s.observation_date AND raw.country = s.country_name
        WHERE c.country_temperature_id IS NULL
        LIMIT 10;
    """,
    'city_to_country': """
        WITH raw_country_keys AS (
            SELECT DISTINCT dt, country FROM staging_country
        )
        SELECT
            ci.observation_date, ci.city_id, ci.city_name,
            ci.country_id, ci.country_name,
            CASE WHEN raw.country IS NULL
                THEN 'absent_from_country_source'
                ELSE 'lost_during_country_pipeline'
            END AS unmatched_reason
        FROM vw_city_temperature AS ci
        LEFT JOIN vw_country_temperature AS c
          ON c.date_id = ci.date_id AND c.country_id = ci.country_id
        LEFT JOIN raw_country_keys AS raw
          ON raw.dt = ci.observation_date AND raw.country = ci.country_name
        WHERE c.country_temperature_id IS NULL
        LIMIT 10;
    """,
    'major_city_to_country': """
        WITH raw_country_keys AS (
            SELECT DISTINCT dt, country FROM staging_country
        )
        SELECT
            ci.observation_date, ci.city_id, ci.city_name,
            ci.country_id, ci.country_name,
            CASE WHEN raw.country IS NULL
                THEN 'absent_from_country_source'
                ELSE 'lost_during_country_pipeline'
            END AS unmatched_reason
        FROM vw_major_city_temperature AS ci
        LEFT JOIN vw_country_temperature AS c
          ON c.date_id = ci.date_id AND c.country_id = ci.country_id
        LEFT JOIN raw_country_keys AS raw
          ON raw.dt = ci.observation_date AND raw.country = ci.country_name
        WHERE c.country_temperature_id IS NULL
        LIMIT 10;
    """,
}

# Mẫu ba cấp City–Country–Global minh họa kết quả enrichment theo cùng date key.
ENRICHED_JOIN_SAMPLE_SQL = """
    SELECT
        ci.observation_date, ci.city_name, ci.country_name,
        ci.average_temperature AS city_temperature,
        c.average_temperature AS country_temperature,
        g.land_average_temperature AS global_land_temperature
    FROM vw_city_temperature AS ci
    LEFT JOIN vw_country_temperature AS c
      ON c.date_id = ci.date_id AND c.country_id = ci.country_id
    LEFT JOIN vw_global_temperature AS g ON g.date_id = ci.date_id
    ORDER BY ci.city_temperature_id
    LIMIT 10;
"""


In [ ]:
# Kiểm tra tiên quyết để tránh NameError khó hiểu khi chạy sai thứ tự cell.
import time
from contextlib import closing

import pandas as pd
import psycopg2

# Các biến sau phải tồn tại vì cell này thực thi SQL đã chuẩn bị ở hai cell trên.
required_join_objects = (
    'DB_CONFIG',
    'JOIN_VALIDATION_SQL',
    'UNMATCHED_SAMPLE_SQL',
    'ENRICHED_JOIN_SAMPLE_SQL',
)
missing_join_objects = [
    name for name in required_join_objects if name not in globals()
]
if missing_join_objects:
    raise RuntimeError(
        'Thiếu biến cần thiết: ' + ', '.join(missing_join_objects)
        + '. Hãy chạy các code cell của Mục 10 từ trên xuống.'
    )

# Tách metrics và diagnostic samples để output chính luôn gọn.
join_validation_rows = []
join_unmatched_samples = {}
current_join = 'connect_database'

try:
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            # Timeout ngăn notebook chờ vô hạn nếu database bị lock hoặc query bất thường.
            cursor.execute("SET lock_timeout = '5s'")
            cursor.execute("SET statement_timeout = '120s'")
            for join_name, statement in JOIN_VALIDATION_SQL.items():
                current_join = join_name
                print(f'Đang kiểm tra {join_name} ...')
                join_started_at = time.perf_counter()
                cursor.execute(statement)
                result_columns = [item[0] for item in cursor.description]
                result = dict(zip(result_columns, cursor.fetchone()))
                elapsed_seconds = time.perf_counter() - join_started_at

                # Ép kiểu và tính metric ở Python để quy tắc PASS được nhìn thấy rõ.
                source_rows = int(result['source_rows'])
                joined_rows = int(result['joined_rows'])
                matched_rows = int(result['matched_rows'])
                unmatched_rows = int(result['unmatched_rows'])
                expected_unmatched = int(result['expected_unmatched_rows'])
                unexpected_unmatched = int(result['unexpected_unmatched_rows'])
                row_multiplication = joined_rows - source_rows
                match_rate = (
                    matched_rows / source_rows * 100 if source_rows else 0.0
                )
                status = (
                    'PASS'
                    if row_multiplication == 0 and unexpected_unmatched == 0
                    else 'FAIL'
                )

                join_validation_rows.append({
                    'join': join_name,
                    'source_rows': source_rows,
                    'joined_rows': joined_rows,
                    'matched_rows': matched_rows,
                    'unmatched_rows': unmatched_rows,
                    'expected_unmatched': expected_unmatched,
                    'unexpected_unmatched': unexpected_unmatched,
                    'row_multiplication': row_multiplication,
                    'match_rate_percent': match_rate,
                    'elapsed_seconds': elapsed_seconds,
                    'status': status,
                })
                print(
                    f'Hoàn thành {join_name}: {elapsed_seconds:,.2f}s — {status}'
                )

                # Chỉ quét lấy mẫu khi có lỗi pipeline thực sự.
                if unexpected_unmatched > 0:
                    cursor.execute(UNMATCHED_SAMPLE_SQL[join_name])
                    sample_columns = [item[0] for item in cursor.description]
                    join_unmatched_samples[join_name] = pd.DataFrame(
                        cursor.fetchall(), columns=sample_columns
                    )
                else:
                    join_unmatched_samples[join_name] = pd.DataFrame()

            # Lấy mẫu enrichment sau khi bốn validation chính đã hoàn tất.
            current_join = 'city_country_global_sample'
            cursor.execute(ENRICHED_JOIN_SAMPLE_SQL)
            enriched_columns = [item[0] for item in cursor.description]
            enriched_rows = cursor.fetchall()
except psycopg2.Error as error:
    raise RuntimeError(
        f'Join validation thất bại tại {current_join}.'
    ) from error

# Xuất kết quả chuẩn hóa cho cell trình bày và final audit.
JOIN_VALIDATION = pd.DataFrame(join_validation_rows)
JOIN_UNMATCHED_SAMPLES = join_unmatched_samples
ENRICHED_JOIN_SAMPLE = pd.DataFrame(
    enriched_rows, columns=enriched_columns
)

Đang kiểm tra country_to_global ...
Hoàn thành country_to_global: 0.51s — PASS
Đang kiểm tra state_to_country ...
Hoàn thành state_to_country: 1.47s — PASS
Đang kiểm tra city_to_country ...
Hoàn thành city_to_country: 8.66s — PASS
Đang kiểm tra major_city_to_country ...
Hoàn thành major_city_to_country: 1.99s — PASS


In [ ]:
# Chỉ hiển thị diagnostic sample nếu query validation phát hiện unmatched bất thường.
display(JOIN_VALIDATION)

for join_name, sample_df in JOIN_UNMATCHED_SAMPLES.items():
    if not sample_df.empty:
        print(f'\n{join_name} — unexpected unmatched sample')
        display(sample_df)

print('\nCity–Country–Global enriched sample')
display(ENRICHED_JOIN_SAMPLE)

# Fail fast nếu join làm nhân bản dòng hoặc làm mất khóa vốn tồn tại trong nguồn.
failed_joins = JOIN_VALIDATION.loc[JOIN_VALIDATION['status'] != 'PASS']
if not failed_joins.empty:
    failed_names = ', '.join(failed_joins['join'].tolist())
    raise RuntimeError(
        f'Các phép join chưa đạt validation: {failed_names}'
    )

expected_total = int(JOIN_VALIDATION['expected_unmatched'].sum())
print(f'Expected unmatched từ dữ liệu nguồn: {expected_total:,} dòng.')
print('Cả 4 phép join đều PASS; không có row multiplication hoặc unexpected unmatched.')

,join,source_rows,joined_rows,matched_rows,unmatched_rows,expected_unmatched,unexpected_unmatched,row_multiplication,match_rate_percent,elapsed_seconds,status
0,country_to_global,577462,577462,573762,3700,3700,0,0,99.359265,0.510904,PASS
1,state_to_country,645675,645675,591003,54672,54672,0,0,91.532582,1.468595,PASS
2,city_to_country,5010113,5010113,5009969,144,144,0,0,99.997126,8.661635,PASS
3,major_city_to_country,239177,239177,233506,5671,5671,0,0,97.628953,1.986323,PASS



City–Country–Global enriched sample


,observation_date,city_name,country_name,city_temperature,country_temperature,global_land_temperature
0,1863-01-01,Århus,Denmark,3.265,-30.324,3.034
1,1863-02-01,Århus,Denmark,3.318,-30.599,3.253
2,1863-03-01,Århus,Denmark,2.699,-28.672,4.682
3,1863-04-01,Århus,Denmark,6.713,-21.978,7.931
4,1863-05-01,Århus,Denmark,10.028,-12.825,11.081
5,1863-06-01,Århus,Denmark,15.299,-4.715,12.794
6,1863-07-01,Århus,Denmark,14.846,-1.643,13.765
7,1863-08-01,Århus,Denmark,16.000,-4.721,13.294
8,1863-09-01,Århus,Denmark,12.501,-11.671,11.230
9,1863-10-01,Århus,Denmark,10.089,-20.850,8.235


Expected unmatched từ dữ liệu nguồn: 64,187 dòng.
Cả 4 phép join đều PASS; không có row multiplication hoặc unexpected unmatched.


## 11. Chạy aggregation queries

Mục này tạo lớp dữ liệu tổng hợp có thể đọc lại nhanh: Global theo năm/thập kỷ và Country, State, City, Major City theo năm. `SQL/04_aggregation.sql` materialize kết quả để không phải group hơn 5 triệu dòng City mỗi lần phân tích.

PostgreSQL `AVG()` bỏ qua `NULL`, nên mỗi nhóm giữ `observation_months`, `valid_temperature_months` và `missing_temperature_months` để người đọc biết mức coverage. Validation kiểm tra grain duy nhất và phương trình `valid + missing = observation`.

**Đầu ra của mục:** `AGGREGATION_VALIDATION`, `AGGREGATION_SAMPLES` và sáu `mv_*` views. Đây là dữ liệu tham khảo cho EDA/trend; Notebook 03 phải cleaning ở monthly views trước vì aggregation không còn từng bản ghi tháng và có thể che khuất missing pattern.

In [ ]:
# Chuẩn bị tạo các materialized views tổng hợp sau khi join validation đã PASS.
import time
from contextlib import closing

import pandas as pd
import psycopg2

# Kiểm tra dependency để tránh tạo aggregate từ analytical layer chưa hợp lệ.
if 'DB_CONFIG' not in globals() or 'PROJECT_ROOT' not in globals():
    raise RuntimeError(
        'Chưa có runtime database. Hãy chạy code cell đầu tiên của Mục 10.'
    )
if 'JOIN_VALIDATION' not in globals():
    raise RuntimeError('Chưa có JOIN_VALIDATION. Hãy hoàn thành Mục 10 trước.')
if (JOIN_VALIDATION['status'] != 'PASS').any():
    raise RuntimeError('Join validation chưa PASS toàn bộ; dừng aggregation.')

# SQL được giữ riêng để có thể review và tái sử dụng ngoài notebook.
AGGREGATION_SQL_PATH = PROJECT_ROOT / 'SQL' / '04_aggregation.sql'
if not AGGREGATION_SQL_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy SQL file: {AGGREGATION_SQL_PATH}')

aggregation_sql = AGGREGATION_SQL_PATH.read_text(encoding='utf-8')
if not aggregation_sql.strip():
    raise ValueError('04_aggregation.sql đang rỗng.')
if 'CREATE MATERIALIZED VIEW' not in aggregation_sql.upper():
    raise ValueError('SQL aggregation không chứa CREATE MATERIALIZED VIEW.')

# Với mỗi materialized view, khai báo grain duy nhất và schema đầu ra dự kiến.
EXPECTED_AGGREGATIONS = {
    'mv_global_temperature_yearly': {
        'grain': ('year',),
        'columns': (
            'year', 'observation_months', 'valid_temperature_months',
            'missing_temperature_months', 'average_land_temperature',
            'minimum_land_temperature', 'maximum_land_temperature',
            'average_land_uncertainty', 'average_land_ocean_temperature',
            'average_land_ocean_uncertainty',
        ),
    },
    'mv_global_temperature_decadal': {
        'grain': ('decade',),
        'columns': (
            'decade', 'first_year', 'last_year', 'observation_months',
            'valid_temperature_months', 'missing_temperature_months',
            'average_land_temperature', 'minimum_land_temperature',
            'maximum_land_temperature', 'average_land_uncertainty',
            'average_land_ocean_temperature',
            'average_land_ocean_uncertainty',
        ),
    },
    'mv_country_temperature_yearly': {
        'grain': ('country_id', 'year'),
        'columns': (
            'country_id', 'country_name', 'year', 'observation_months',
            'valid_temperature_months', 'missing_temperature_months',
            'average_temperature', 'minimum_temperature',
            'maximum_temperature', 'average_temperature_uncertainty',
        ),
    },
    'mv_state_temperature_yearly': {
        'grain': ('state_id', 'year'),
        'columns': (
            'state_id', 'state_name', 'country_id', 'country_name', 'year',
            'observation_months', 'valid_temperature_months',
            'missing_temperature_months', 'average_temperature',
            'minimum_temperature', 'maximum_temperature',
            'average_temperature_uncertainty',
        ),
    },
    'mv_city_temperature_yearly': {
        'grain': ('city_id', 'year'),
        'columns': (
            'city_id', 'city_name', 'country_id', 'country_name',
            'latitude', 'longitude', 'is_major_city', 'year',
            'observation_months', 'valid_temperature_months',
            'missing_temperature_months', 'average_temperature',
            'minimum_temperature', 'maximum_temperature',
            'average_temperature_uncertainty',
        ),
    },
    'mv_major_city_temperature_yearly': {
        'grain': ('city_id', 'year'),
        'columns': (
            'city_id', 'city_name', 'country_id', 'country_name',
            'latitude', 'longitude', 'is_major_city', 'year',
            'observation_months', 'valid_temperature_months',
            'missing_temperature_months', 'average_temperature',
            'minimum_temperature', 'maximum_temperature',
            'average_temperature_uncertainty',
        ),
    },
}

print(f'Aggregation SQL: {AGGREGATION_SQL_PATH.name}')
print(f'Materialized views dự kiến: {len(EXPECTED_AGGREGATIONS)}')

Aggregation SQL: 04_aggregation.sql
Materialized views dự kiến: 6


In [ ]:
# Build và schema validation được bao trong một transaction để rollback đồng bộ.
aggregation_started_at = time.perf_counter()
current_aggregation_step = 'connect_database'
connection = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    with connection.cursor() as cursor:
        # Giới hạn thời gian chờ lock/query vì City aggregation phải quét nhiều triệu dòng.
        cursor.execute("SET lock_timeout = '5s'")
        cursor.execute("SET statement_timeout = '180s'")
        current_aggregation_step = 'execute_04_aggregation.sql'
        cursor.execute(aggregation_sql)

        cursor.execute(
            """
            SELECT matviewname
            FROM pg_matviews
            WHERE schemaname = 'public'
              AND matviewname = ANY(%s);
            """,
            (list(EXPECTED_AGGREGATIONS),),
        )
        available_aggregations = {row[0] for row in cursor.fetchall()}

        # Đọc pg_catalog để kiểm tra loại relation và đúng thứ tự cột của từng MV.
        aggregation_view_rows = []
        for view_name, config in EXPECTED_AGGREGATIONS.items():
            current_aggregation_step = f'validate_schema:{view_name}'
            cursor.execute(
                """
                SELECT attribute.attname AS column_name
                FROM pg_attribute AS attribute
                JOIN pg_class AS relation
                  ON relation.oid = attribute.attrelid
                JOIN pg_namespace AS namespace
                  ON namespace.oid = relation.relnamespace
                WHERE namespace.nspname = 'public'
                  AND relation.relname = %s
                  AND relation.relkind = 'm'
                  AND attribute.attnum > 0
                  AND NOT attribute.attisdropped
                ORDER BY attribute.attnum;
                """,
                (view_name,),
            )
            actual_columns = tuple(row[0] for row in cursor.fetchall())
            exists = view_name in available_aggregations
            columns_ok = actual_columns == config['columns']
            aggregation_view_rows.append({
                'materialized_view': view_name,
                'exists': exists,
                'column_count': len(actual_columns),
                'columns_ok': columns_ok,
                'status': 'PASS' if exists and columns_ok else 'FAIL',
            })

    # Chỉ giữ các materialized views khi cả sáu schema đều đúng contract.
    connection.commit()
except Exception as error:
    if connection is not None:
        connection.rollback()
    raise RuntimeError(
        f'Aggregation build thất bại tại {current_aggregation_step}. '
        'Transaction đã rollback.'
    ) from error
finally:
    if connection is not None:
        connection.close()

# Tách build validation khỏi quality validation ở cell kế tiếp.
AGGREGATION_VIEW_VALIDATION = pd.DataFrame(aggregation_view_rows)
AGGREGATION_BUILD_SECONDS = time.perf_counter() - aggregation_started_at
display(AGGREGATION_VIEW_VALIDATION)

failed_aggregation_views = AGGREGATION_VIEW_VALIDATION.loc[
    AGGREGATION_VIEW_VALIDATION['status'] != 'PASS'
]
if not failed_aggregation_views.empty:
    failed_names = ', '.join(
        failed_aggregation_views['materialized_view'].tolist()
    )
    raise RuntimeError(f'Aggregation views chưa hợp lệ: {failed_names}')

print(
    f'Đã tạo và xác nhận 6 materialized views trong '
    f'{AGGREGATION_BUILD_SECONDS:,.2f} giây.'
)

,materialized_view,exists,column_count,columns_ok,status
0,mv_global_temperature_yearly,True,10,True,PASS
1,mv_global_temperature_decadal,True,12,True,PASS
2,mv_country_temperature_yearly,True,10,True,PASS
3,mv_state_temperature_yearly,True,12,True,PASS
4,mv_city_temperature_yearly,True,15,True,PASS
5,mv_major_city_temperature_yearly,True,15,True,PASS


Đã tạo và xác nhận 6 materialized views trong 10.33 giây.


In [ ]:
# Kiểm tra chất lượng aggregate: grain, khóa NULL và monthly coverage.
from psycopg2 import sql

aggregation_validation_rows = []
aggregation_samples = {}
current_aggregation_view = 'connect_database'

try:
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute("SET lock_timeout = '5s'")
            cursor.execute("SET statement_timeout = '120s'")

            # Dựng SQL động bằng Identifier để an toàn với tên view/cột.
            for view_name, config in EXPECTED_AGGREGATIONS.items():
                current_aggregation_view = view_name
                grain_identifiers = [
                    sql.Identifier(column) for column in config['grain']
                ]
                grain_sql = sql.SQL(', ').join(grain_identifiers)
                null_grain_sql = sql.SQL(' OR ').join(
                    sql.SQL('{} IS NULL').format(identifier)
                    for identifier in grain_identifiers
                )

                # Mỗi grain phải duy nhất; valid + missing phải bằng tổng số tháng quan sát.
                validation_statement = sql.SQL(
                    """
                    SELECT
                        COUNT(*)::BIGINT AS row_count,
                        (COUNT(*) - COUNT(DISTINCT ({})))::BIGINT
                            AS duplicate_grain_rows,
                        COUNT(*) FILTER (WHERE {})::BIGINT
                            AS null_grain_rows,
                        COUNT(*) FILTER (
                            WHERE observation_months <= 0
                               OR valid_temperature_months < 0
                               OR missing_temperature_months < 0
                               OR valid_temperature_months
                                  + missing_temperature_months
                                  <> observation_months
                        )::BIGINT AS invalid_coverage_rows
                    FROM {};
                    """
                ).format(
                    grain_sql, null_grain_sql, sql.Identifier(view_name)
                )
                started_at = time.perf_counter()
                cursor.execute(validation_statement)
                (
                    row_count, duplicate_grain_rows,
                    null_grain_rows, invalid_coverage_rows,
                ) = cursor.fetchone()
                elapsed_seconds = time.perf_counter() - started_at

                status = (
                    'PASS'
                    if row_count > 0
                    and duplicate_grain_rows == 0
                    and null_grain_rows == 0
                    and invalid_coverage_rows == 0
                    else 'FAIL'
                )
                aggregation_validation_rows.append({
                    'materialized_view': view_name,
                    'row_count': int(row_count),
                    'duplicate_grain_rows': int(duplicate_grain_rows),
                    'null_grain_rows': int(null_grain_rows),
                    'invalid_coverage_rows': int(invalid_coverage_rows),
                    'validation_seconds': elapsed_seconds,
                    'status': status,
                })

                # Lấy 5 dòng có thứ tự ổn định để người đọc kiểm tra nhanh output.
                sample_statement = sql.SQL(
                    'SELECT * FROM {} ORDER BY {} LIMIT 5'
                ).format(sql.Identifier(view_name), grain_sql)
                cursor.execute(sample_statement)
                sample_columns = [item[0] for item in cursor.description]
                aggregation_samples[view_name] = pd.DataFrame(
                    cursor.fetchall(), columns=sample_columns
                )
except psycopg2.Error as error:
    raise RuntimeError(
        f'Aggregation validation thất bại tại {current_aggregation_view}.'
    ) from error

# Công bố kết quả cho Mục 12 và final audit chỉ sau khi hoàn tất sáu view.
AGGREGATION_VALIDATION = pd.DataFrame(aggregation_validation_rows)
AGGREGATION_SAMPLES = aggregation_samples
display(AGGREGATION_VALIDATION)

for view_name, sample_df in AGGREGATION_SAMPLES.items():
    print(f'\n{view_name} — 5 dòng mẫu')
    display(sample_df)

failed_aggregations = AGGREGATION_VALIDATION.loc[
    AGGREGATION_VALIDATION['status'] != 'PASS'
]
if not failed_aggregations.empty:
    failed_names = ', '.join(
        failed_aggregations['materialized_view'].tolist()
    )
    raise RuntimeError(f'Aggregation validation chưa PASS: {failed_names}')

print('Cả 6 aggregation outputs đều PASS về grain và monthly coverage.')

,materialized_view,row_count,duplicate_grain_rows,null_grain_rows,invalid_coverage_rows,validation_seconds,status
0,mv_global_temperature_yearly,266,0,0,0,0.002942,PASS
1,mv_global_temperature_decadal,27,0,0,0,0.000811,PASS
2,mv_country_temperature_yearly,48243,0,0,0,0.116987,PASS
3,mv_state_temperature_yearly,53975,0,0,0,0.054725,PASS
4,mv_city_temperature_yearly,418208,0,0,0,0.335969,PASS
5,mv_major_city_temperature_yearly,19976,0,0,0,0.011579,PASS



mv_global_temperature_yearly — 5 dòng mẫu


,year,observation_months,valid_temperature_months,missing_temperature_months,average_land_temperature,minimum_land_temperature,maximum_land_temperature,average_land_uncertainty,average_land_ocean_temperature,average_land_ocean_uncertainty
0,1750,12,11,1,8.719364,2.772,15.868,2.637818,None,None
1,1751,12,7,5,7.976143,0.963,14.405,2.781143,None,None
2,1752,12,6,6,5.779833,0.348,8.265,2.977000,None,None
3,1753,12,12,0,8.388083,0.559,15.092,3.176000,None,None
4,1754,12,12,0,8.469333,-1.249,14.681,3.494250,None,None



mv_global_temperature_decadal — 5 dòng mẫu


,decade,first_year,last_year,observation_months,valid_temperature_months,missing_temperature_months,average_land_temperature,minimum_land_temperature,maximum_land_temperature,average_land_uncertainty,average_land_ocean_temperature,average_land_ocean_uncertainty
0,1750,1750,1759,120,108,12,8.149852,-1.503,17.910,3.375509,None,None
1,1760,1760,1769,120,120,0,7.981625,-2.080,19.021,2.858717,None,None
2,1770,1770,1779,120,120,0,8.400108,-0.806,16.521,2.663092,None,None
3,1780,1780,1789,120,120,0,8.141392,-1.431,16.468,2.394250,None,None
4,1790,1790,1799,120,120,0,8.336867,0.348,14.951,1.916067,None,None


...


mv_city_temperature_yearly — 5 dòng mẫu


,city_id,city_name,country_id,country_name,latitude,longitude,is_major_city,year,observation_months,valid_temperature_months,missing_temperature_months,average_temperature,minimum_temperature,maximum_temperature,average_temperature_uncertainty
0,1,A Coruña,208,Spain,42.59,-8.73,False,1863,12,12,0,13.008833,8.131,18.898,1.436250
1,1,A Coruña,208,Spain,42.59,-8.73,False,1864,12,12,0,13.509083,7.588,20.097,1.295417
2,1,A Coruña,208,Spain,42.59,-8.73,False,1865,12,12,0,13.405667,8.242,19.803,1.180417
3,1,A Coruña,208,Spain,42.59,-8.73,False,1866,12,12,0,13.218667,8.793,18.993,0.952583
4,1,A Coruña,208,Spain,42.59,-8.73,False,1867,12,12,0,13.210417,7.759,18.225,1.088333



mv_major_city_temperature_yearly — 5 dòng mẫu


,city_id,city_name,country_id,country_name,latitude,longitude,is_major_city,year,observation_months,valid_temperature_months,missing_temperature_months,average_temperature,minimum_temperature,maximum_temperature,average_temperature_uncertainty
0,11,Abidjan,50,Côte D'Ivoire,5.63,-3.23,True,1849,12,12,0,25.582583,23.576,28.101,1.388583
1,11,Abidjan,50,Côte D'Ivoire,5.63,-3.23,True,1850,12,12,0,25.518250,23.758,27.890,1.503667
2,11,Abidjan,50,Côte D'Ivoire,5.63,-3.23,True,1851,12,12,0,25.672333,23.752,27.363,1.443583
3,11,Abidjan,50,Côte D'Ivoire,5.63,-3.23,True,1852,12,0,12,NaN,NaN,NaN,NaN
4,11,Abidjan,50,Côte D'Ivoire,5.63,-3.23,True,1853,12,0,12,NaN,NaN,NaN,NaN


Cả 6 aggregation outputs đều PASS về grain và monthly coverage.


## 12. Tạo indexes và kiểm tra query plan

Mục này tối ưu cách PostgreSQL truy cập dữ liệu nhưng không thay đổi bất kỳ giá trị nào. Notebook lưu baseline của bốn truy vấn, chạy `SQL/05_indexes.sql`, xác nhận đúng bảng/thứ tự cột/unique rồi đo lại bằng `EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)`.

Primary key và unique constraints đã có index nên không lặp lại. Index mới phục vụ lịch sử theo geography, Country–Date và grain aggregation. `Seq Scan` không mặc định là lỗi: truy vấn validation toàn bảng vẫn có thể đọc tuần tự vì đó là kế hoạch tối ưu. So sánh thời gian chịu ảnh hưởng của cache nên notebook lưu cả plan, cost và buffers.

**Đầu ra của mục:** `INDEX_VALIDATION`, `BASELINE_QUERY_PLANS`, `POST_INDEX_QUERY_PLANS` và `INDEX_PLAN_COMPARISON`. Notebook 03 nhận cùng dữ liệu như trước, chỉ truy vấn nhanh hơn; nếu Mục 11 được rebuild thì phải chạy lại Mục 12 vì index của materialized views sẽ bị xóa cùng view.

In [ ]:
# Đo execution plan trước/sau và xác nhận 13 indexes đúng metadata dự kiến.
import time
from contextlib import closing

import pandas as pd
import psycopg2

# Index build chỉ bắt đầu khi toàn bộ materialized views đã vượt quality checks.
if 'AGGREGATION_VALIDATION' not in globals():
    raise RuntimeError('Chưa có AGGREGATION_VALIDATION. Hãy chạy Mục 11 trước.')
if (AGGREGATION_VALIDATION['status'] != 'PASS').any():
    raise RuntimeError('Aggregation validation chưa PASS; dừng index build.')

# Đọc script index có IF NOT EXISTS để cell an toàn khi chạy lại.
INDEX_SQL_PATH = PROJECT_ROOT / 'SQL' / '05_indexes.sql'
if not INDEX_SQL_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy SQL file: {INDEX_SQL_PATH}')
index_sql = INDEX_SQL_PATH.read_text(encoding='utf-8')
if not index_sql.strip():
    raise ValueError('05_indexes.sql đang rỗng.')
if 'CREATE INDEX IF NOT EXISTS' not in index_sql.upper():
    raise ValueError('Index SQL chưa dùng cơ chế IF NOT EXISTS.')

# Contract gồm bảng đích, thứ tự cột và thuộc tính unique của từng index.
EXPECTED_INDEXES = {
    'idx_staging_country_country_dt': {
        'table': 'staging_country', 'columns': ('country', 'dt'),
        'unique': False,
    },
    'idx_fact_country_temperature_country_date': {
        'table': 'fact_country_temperature',
        'columns': ('country_id', 'date_id'), 'unique': False,
    },
    'idx_fact_state_temperature_state_date': {
        'table': 'fact_state_temperature',
        'columns': ('state_id', 'date_id'), 'unique': False,
    },
    'idx_fact_city_temperature_city_date': {
        'table': 'fact_city_temperature',
        'columns': ('city_id', 'date_id'), 'unique': False,
    },
    'idx_fact_major_city_temperature_city_date': {
        'table': 'fact_major_city_temperature',
        'columns': ('city_id', 'date_id'), 'unique': False,
    },
    'idx_dim_state_country': {
        'table': 'dim_state', 'columns': ('country_id',), 'unique': False,
    },
    'idx_dim_city_country_major': {
        'table': 'dim_city',
        'columns': ('country_id', 'is_major_city'), 'unique': False,
    },
    'ux_mv_global_temperature_yearly_year': {
        'table': 'mv_global_temperature_yearly',
        'columns': ('year',), 'unique': True,
    },
    'ux_mv_global_temperature_decadal_decade': {
        'table': 'mv_global_temperature_decadal',
        'columns': ('decade',), 'unique': True,
    },
    'ux_mv_country_temperature_yearly_grain': {
        'table': 'mv_country_temperature_yearly',
        'columns': ('country_id', 'year'), 'unique': True,
    },
    'ux_mv_state_temperature_yearly_grain': {
        'table': 'mv_state_temperature_yearly',
        'columns': ('state_id', 'year'), 'unique': True,
    },
    'ux_mv_city_temperature_yearly_grain': {
        'table': 'mv_city_temperature_yearly',
        'columns': ('city_id', 'year'), 'unique': True,
    },
    'ux_mv_major_city_temperature_yearly_grain': {
        'table': 'mv_major_city_temperature_yearly',
        'columns': ('city_id', 'year'), 'unique': True,
    },
}

# Ghi nhận index đã tồn tại để không diễn giải sai baseline của lần chạy lại.
with closing(psycopg2.connect(**DB_CONFIG)) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        # Ghi nhận chính xác các index Mục 12 đã có trước benchmark.
        cursor.execute(
            """
            SELECT indexname FROM pg_indexes
            WHERE schemaname = 'public' AND indexname = ANY(%s);
            """,
            (list(EXPECTED_INDEXES),),
        )
        INDEXES_EXISTING_BEFORE = {row[0] for row in cursor.fetchall()}

        # Chọn country/city có nhiều năm nhất làm tham số đại diện cho benchmark.
        cursor.execute(
            """
            SELECT country_id FROM mv_country_temperature_yearly
            GROUP BY country_id ORDER BY COUNT(*) DESC, country_id LIMIT 1;
            """
        )
        sample_country_id = cursor.fetchone()[0]
        cursor.execute(
            """
            SELECT city_id FROM mv_city_temperature_yearly
            GROUP BY city_id ORDER BY COUNT(*) DESC, city_id LIMIT 1;
            """
        )
        sample_city_id = cursor.fetchone()[0]

BASELINE_IS_PRE_INDEX = not INDEXES_EXISTING_BEFORE
if BASELINE_IS_PRE_INDEX:
    print('Baseline hợp lệ: chưa có index bổ sung của Mục 12.')
else:
    print(
        f'Baseline hiện tại đã có {len(INDEXES_EXISTING_BEFORE)} index '
        'của Mục 12; so sánh lần chạy lại chỉ mang tính tham khảo.'
    )

# Bốn workload đại diện cho truy vấn lịch sử monthly và yearly thường dùng.
PLAN_QUERY_CONFIG = {
    'country_history': {
        'statement': (
            'SELECT date_id, average_temperature '
            'FROM fact_country_temperature WHERE country_id = %s '
            'ORDER BY date_id'
        ),
        'params': (sample_country_id,),
    },
    'city_history': {
        'statement': (
            'SELECT date_id, average_temperature '
            'FROM fact_city_temperature WHERE city_id = %s '
            'ORDER BY date_id'
        ),
        'params': (sample_city_id,),
    },
    'country_yearly': {
        'statement': (
            'SELECT * FROM mv_country_temperature_yearly '
            'WHERE country_id = %s ORDER BY year'
        ),
        'params': (sample_country_id,),
    },
    'city_yearly': {
        'statement': (
            'SELECT * FROM mv_city_temperature_yearly '
            'WHERE city_id = %s ORDER BY year'
        ),
        'params': (sample_city_id,),
    },
}


# EXPLAIN JSON có cấu trúc cây; hàm đệ quy gom mọi node để tìm index scan.
def collect_plan_nodes(plan_node: dict) -> list[dict]:
    nodes = [plan_node]
    for child in plan_node.get('Plans', []):
        nodes.extend(collect_plan_nodes(child))
    return nodes


# Chạy EXPLAIN ANALYZE readonly và rút các metric có thể so sánh giữa hai thời điểm.
def capture_query_plans(stage: str) -> pd.DataFrame:
    plan_rows = []
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            cursor.execute("SET lock_timeout = '5s'")
            cursor.execute("SET statement_timeout = '120s'")
            for query_name, config in PLAN_QUERY_CONFIG.items():
                cursor.execute(
                    'EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) '
                    + config['statement'],
                    config['params'],
                )
                plan_document = cursor.fetchone()[0][0]
                root_plan = plan_document['Plan']
                nodes = collect_plan_nodes(root_plan)
                node_types = list(dict.fromkeys(
                    node['Node Type'] for node in nodes
                ))
                index_names = list(dict.fromkeys(
                    node['Index Name'] for node in nodes
                    if node.get('Index Name')
                ))
                plan_rows.append({
                    'stage': stage,
                    'query': query_name,
                    'node_types': ' → '.join(node_types),
                    'uses_index': bool(index_names),
                    'index_names': ', '.join(index_names),
                    'total_cost': float(root_plan['Total Cost']),
                    'actual_rows': int(root_plan['Actual Rows']),
                    'planning_time_ms': float(plan_document['Planning Time']),
                    'execution_time_ms': float(plan_document['Execution Time']),
                    'shared_hit_blocks': int(
                        root_plan.get('Shared Hit Blocks', 0)
                    ),
                    'shared_read_blocks': int(
                        root_plan.get('Shared Read Blocks', 0)
                    ),
                })
    return pd.DataFrame(plan_rows)


# Baseline phản ánh trạng thái thật hiện tại; xem BASELINE_IS_PRE_INDEX trước khi kết luận.
BASELINE_QUERY_PLANS = capture_query_plans('before_index')
display(BASELINE_QUERY_PLANS)

Baseline hợp lệ: chưa có index bổ sung của Mục 12.


,stage,query,node_types,uses_index,index_names,total_cost,actual_rows,planning_time_ms,execution_time_ms,shared_hit_blocks,shared_read_blocks
0,before_index,country_history,Gather Merge → Sort → Seq Scan,False,,9698.07,3239,1.522,52.995,735,4674
1,before_index,city_history,Gather Merge → Sort → Seq Scan,False,,77469.12,1809,1.770,168.662,6475,40386
2,before_index,country_yearly,Sort → Seq Scan,False,,1160.38,271,0.673,2.864,643,0
3,before_index,city_yearly,Gather Merge → Sort → Seq Scan,False,,10508.78,151,0.551,46.653,2404,5222


In [ ]:
# Tạo indexes và kiểm tra catalog trong một transaction có rollback khi lỗi.
index_build_started_at = time.perf_counter()
current_index_step = 'connect_database'
connection = None

try:
    connection = psycopg2.connect(**DB_CONFIG)
    with connection.cursor() as cursor:
        # Index trên bảng City lớn cần timeout rộng hơn nhưng vẫn không chờ vô hạn.
        cursor.execute("SET lock_timeout = '5s'")
        cursor.execute("SET statement_timeout = '300s'")
        current_index_step = 'execute_05_indexes.sql'
        cursor.execute(index_sql)

        # Đọc pg_index/pg_attribute để xác nhận cả thứ tự cột và unique flag.
        index_validation_rows = []
        for index_name, expected in EXPECTED_INDEXES.items():
            current_index_step = f'validate:{index_name}'
            cursor.execute(
                """
                SELECT
                    table_relation.relname AS table_name,
                    index_info.indisunique AS is_unique,
                    ARRAY(
                        SELECT attribute.attname
                        FROM unnest(index_info.indkey) WITH ORDINALITY
                             AS index_key(attnum, position)
                        JOIN pg_attribute AS attribute
                          ON attribute.attrelid = table_relation.oid
                         AND attribute.attnum = index_key.attnum
                        ORDER BY index_key.position
                    ) AS indexed_columns
                FROM pg_class AS index_relation
                JOIN pg_index AS index_info
                  ON index_info.indexrelid = index_relation.oid
                JOIN pg_class AS table_relation
                  ON table_relation.oid = index_info.indrelid
                JOIN pg_namespace AS namespace
                  ON namespace.oid = table_relation.relnamespace
                WHERE namespace.nspname = 'public'
                  AND index_relation.relname = %s;
                """,
                (index_name,),
            )
            result = cursor.fetchone()
            exists = result is not None
            if exists:
                actual_table, actual_unique, actual_columns = result
                actual_columns = tuple(actual_columns)
            else:
                actual_table, actual_unique, actual_columns = None, None, ()

            table_ok = actual_table == expected['table']
            columns_ok = actual_columns == expected['columns']
            unique_ok = actual_unique == expected['unique']
            status = (
                'PASS'
                if exists and table_ok and columns_ok and unique_ok
                else 'FAIL'
            )
            index_validation_rows.append({
                'index': index_name,
                'table': actual_table,
                'columns': actual_columns,
                'is_unique': actual_unique,
                'table_ok': table_ok,
                'columns_ok': columns_ok,
                'unique_ok': unique_ok,
                'status': status,
            })

        failed_index_rows = [
            row for row in index_validation_rows if row['status'] != 'PASS'
        ]
        if failed_index_rows:
            failed_names = ', '.join(row['index'] for row in failed_index_rows)
            raise RuntimeError(f'Index validation thất bại: {failed_names}')

    # Chỉ commit khi toàn bộ 13 index khớp contract.
    connection.commit()
except Exception as error:
    if connection is not None:
        connection.rollback()
    raise RuntimeError(
        f'Index build thất bại tại {current_index_step}. '
        'Transaction đã rollback.'
    ) from error
finally:
    if connection is not None:
        connection.close()

# Lưu kết quả để đo post-index plan và đưa vào final audit.
INDEX_VALIDATION = pd.DataFrame(index_validation_rows)
INDEX_BUILD_SECONDS = time.perf_counter() - index_build_started_at
display(INDEX_VALIDATION)
print(
    f'Đã tạo/xác nhận {len(INDEX_VALIDATION)} indexes trong '
    f'{INDEX_BUILD_SECONDS:,.2f} giây.'
)

,index,table,columns,is_unique,table_ok,columns_ok,unique_ok,status
0,idx_staging_country_country_dt,staging_country,"(country, dt)",False,True,True,True,PASS
1,idx_fact_country_temperature_country_date,fact_country_temperature,"(country_id, date_id)",False,True,True,True,PASS
2,idx_fact_state_temperature_state_date,fact_state_temperature,"(state_id, date_id)",False,True,True,True,PASS
3,idx_fact_city_temperature_city_date,fact_city_temperature,"(city_id, date_id)",False,True,True,True,PASS
4,idx_fact_major_city_temperature_city_date,fact_major_city_temperature,"(city_id, date_id)",False,True,True,True,PASS
5,idx_dim_state_country,dim_state,"(country_id,)",False,True,True,True,PASS
6,idx_dim_city_country_major,dim_city,"(country_id, is_major_city)",False,True,True,True,PASS
7,ux_mv_global_temperature_yearly_year,mv_global_temperature_yearly,"(year,)",True,True,True,True,PASS
8,ux_mv_global_temperature_decadal_decade,mv_global_temperature_decadal,"(decade,)",True,True,True,True,PASS
9,ux_mv_country_temperature_yearly_grain,mv_country_temperature_yearly,"(country_id, year)",True,True,True,True,PASS


Đã tạo/xác nhận 13 indexes trong 6.92 giây.


In [ ]:
# Không benchmark nếu metadata index chưa PASS để tránh kết luận hiệu năng sai.
if (INDEX_VALIDATION['status'] != 'PASS').any():
    raise RuntimeError('Index validation chưa PASS; không thể đo post-index plan.')

# Chạy lại đúng bốn workload với cùng tham số sau khi tạo index.
POST_INDEX_QUERY_PLANS = capture_query_plans('after_index')

# Đổi hậu tố before/after rồi merge theo tên query để so sánh cùng một hàng.
before_metrics = BASELINE_QUERY_PLANS.rename(columns={
    'node_types': 'node_types_before',
    'uses_index': 'uses_index_before',
    'index_names': 'index_names_before',
    'total_cost': 'total_cost_before',
    'execution_time_ms': 'execution_time_ms_before',
    'shared_hit_blocks': 'shared_hit_blocks_before',
    'shared_read_blocks': 'shared_read_blocks_before',
})[[
    'query', 'node_types_before', 'uses_index_before',
    'index_names_before', 'total_cost_before',
    'execution_time_ms_before', 'shared_hit_blocks_before',
    'shared_read_blocks_before',
]]
after_metrics = POST_INDEX_QUERY_PLANS.rename(columns={
    'node_types': 'node_types_after',
    'uses_index': 'uses_index_after',
    'index_names': 'index_names_after',
    'total_cost': 'total_cost_after',
    'execution_time_ms': 'execution_time_ms_after',
    'shared_hit_blocks': 'shared_hit_blocks_after',
    'shared_read_blocks': 'shared_read_blocks_after',
})[[
    'query', 'node_types_after', 'uses_index_after',
    'index_names_after', 'total_cost_after',
    'execution_time_ms_after', 'shared_hit_blocks_after',
    'shared_read_blocks_after',
]]

INDEX_PLAN_COMPARISON = before_metrics.merge(after_metrics, on='query')
# Tính speedup và cost reduction; replace(0) tránh phép chia cho 0.
INDEX_PLAN_COMPARISON['speedup_ratio'] = (
    INDEX_PLAN_COMPARISON['execution_time_ms_before']
    / INDEX_PLAN_COMPARISON['execution_time_ms_after'].replace(0, pd.NA)
)
INDEX_PLAN_COMPARISON['cost_reduction_percent'] = (
    (INDEX_PLAN_COMPARISON['total_cost_before']
     - INDEX_PLAN_COMPARISON['total_cost_after'])
    / INDEX_PLAN_COMPARISON['total_cost_before'].replace(0, pd.NA)
    * 100
)
INDEX_PLAN_COMPARISON['plan_changed'] = (
    INDEX_PLAN_COMPARISON['node_types_before']
    != INDEX_PLAN_COMPARISON['node_types_after']
)

display(POST_INDEX_QUERY_PLANS)
display(INDEX_PLAN_COMPARISON[[
    'query', 'uses_index_before', 'uses_index_after',
    'index_names_after', 'execution_time_ms_before',
    'execution_time_ms_after', 'speedup_ratio',
    'total_cost_before', 'total_cost_after',
    'cost_reduction_percent', 'plan_changed',
]])

# Chỉ diễn giải index adoption như tác động lần build khi baseline thật sự chưa có index.
if BASELINE_IS_PRE_INDEX:
    index_adoption_count = int(
        (~INDEX_PLAN_COMPARISON['uses_index_before']
         & INDEX_PLAN_COMPARISON['uses_index_after']).sum()
    )
    print(
        f'{index_adoption_count}/{len(INDEX_PLAN_COMPARISON)} truy vấn '
        'đã chuyển từ không dùng index sang dùng index.'
    )
else:
    print(
        'Các index đã tồn tại trước baseline; không diễn giải speedup '
        'như tác động nhân quả của lần chạy này.'
    )

print('Mục 12 hoàn tất: toàn bộ index validation đều PASS.')

,stage,query,node_types,uses_index,index_names,total_cost,actual_rows,planning_time_ms,execution_time_ms,shared_hit_blocks,shared_read_blocks
0,after_index,country_history,Index Scan,True,idx_fact_country_temperature_country_date,4555.41,3239,1.971,0.927,3,47
1,after_index,city_history,Index Scan,True,idx_fact_city_temperature_city_date,3313.10,1809,1.643,0.881,13,15
2,after_index,country_yearly,Sort → Bitmap Heap Scan → Bitmap Index Scan,True,ux_mv_country_temperature_yearly_grain,512.11,271,1.328,0.535,189,3
3,after_index,city_yearly,Index Scan,True,ux_mv_city_temperature_yearly_grain,271.17,151,0.753,0.090,0,6


,query,uses_index_before,uses_index_after,index_names_after,execution_time_ms_before,execution_time_ms_after,speedup_ratio,total_cost_before,total_cost_after,cost_reduction_percent,plan_changed
0,country_history,False,True,idx_fact_country_temperature_country_date,52.995,0.927,57.168285,9698.07,4555.41,53.027664,True
1,city_history,False,True,idx_fact_city_temperature_city_date,168.662,0.881,191.443814,77469.12,3313.10,95.723328,True
2,country_yearly,False,True,ux_mv_country_temperature_yearly_grain,2.864,0.535,5.353271,1160.38,512.11,55.867044,True
3,city_yearly,False,True,ux_mv_city_temperature_yearly_grain,46.653,0.090,518.366667,10508.78,271.17,97.419586,True


4/4 truy vấn đã chuyển từ không dùng index sang dùng index.
Mục 12 hoàn tất: toàn bộ index validation đều PASS.


## 13. Kết luận và chuyển giao cho Notebook 03

Mục cuối thực hiện final audit trước khi bàn giao database. Pipeline chỉ được đánh dấu `READY` khi tất cả checkpoint từ Mục 4–12, kiểm kê đối tượng PostgreSQL và row count của năm monthly views đều `PASS`.

Notebook 02 kết thúc ở dữ liệu đã được tổ chức và kiểm chứng, chưa phải dữ liệu cleaned. Người làm Notebook 03 sẽ restore bản backup vào PostgreSQL trên máy họ, đọc monthly views và tự tài liệu hóa cách xử lý missing values, duplicates và outliers.

### a. Final audit và data contract

Hai cell dưới đây chỉ đọc database. Cell đầu tổng hợp các validation thành `PIPELINE_READINESS`; cell sau tạo `DATABASE_OBJECT_INVENTORY` và `HANDOFF_MANIFEST` để người nhận biết chính xác view, grain, lineage và row count cần kiểm tra sau restore.

In [65]:
# Tổng hợp toàn bộ checkpoint Mục 4–12 thành một readiness report duy nhất.
from contextlib import closing

import pandas as pd
import psycopg2
from psycopg2 import sql

# Final audit phụ thuộc vào các DataFrame validation đã tạo ở những mục trước.
required_final_objects = (
    'DB_CONFIG', 'SOURCE_ROW_COUNTS', 'input_check_df',
    'ROW_COUNT_VALIDATION', 'FACT_LOAD_VALIDATION',
    'VIEW_VALIDATION', 'JOIN_VALIDATION',
    'AGGREGATION_VALIDATION', 'INDEX_VALIDATION',
    'EXPECTED_INDEXES',
)
missing_final_objects = [
    name for name in required_final_objects if name not in globals()
]
if missing_final_objects:
    raise RuntimeError(
        'Thiếu kết quả pipeline: ' + ', '.join(missing_final_objects)
        + '. Hãy chạy các mục tương ứng trước Mục 13.'
    )

# Chuyển điều kiện input contract thành số lượng lỗi để dùng chung format checkpoint.
input_failures = int((
    (~input_check_df['exists'])
    | (input_check_df['size_mb'] <= 0)
    | (~input_check_df['header_ok'])
).sum())

# Mỗi tuple mô tả tên checkpoint, số đối tượng kiểm tra, số lỗi và ý nghĩa.
checkpoint_config = (
    (
        'input_files', len(input_check_df), input_failures,
        'CSV tồn tại, không rỗng và đúng header Notebook 01',
    ),
    (
        'staging_load', len(ROW_COUNT_VALIDATION),
        int((ROW_COUNT_VALIDATION['status'] != 'PASS').sum()),
        'Row count và metadata staging khớp nguồn',
    ),
    (
        'fact_load', len(FACT_LOAD_VALIDATION),
        int((FACT_LOAD_VALIDATION['status'] != 'PASS').sum()),
        'Fact row count khớp staging',
    ),
    (
        'analytical_views', len(VIEW_VALIDATION),
        int((VIEW_VALIDATION['status'] != 'PASS').sum()),
        'Monthly views tồn tại và đúng schema',
    ),
    (
        'cross_level_joins', len(JOIN_VALIDATION),
        int((JOIN_VALIDATION['status'] != 'PASS').sum()),
        'Không có row multiplication hoặc unexpected unmatched',
    ),
    (
        'aggregations', len(AGGREGATION_VALIDATION),
        int((AGGREGATION_VALIDATION['status'] != 'PASS').sum()),
        'Aggregation grain và monthly coverage hợp lệ',
    ),
    (
        'indexes', len(INDEX_VALIDATION),
        int((INDEX_VALIDATION['status'] != 'PASS').sum()),
        'Index đúng bảng, cột và unique property',
    ),
)

# Chuẩn hóa các kết quả khác nhau thành cùng schema PASS/FAIL.
readiness_rows = []
for checkpoint, checked_objects, failed_objects, meaning in checkpoint_config:
    readiness_rows.append({
        'checkpoint': checkpoint,
        'checked_objects': int(checked_objects),
        'passed_objects': int(checked_objects - failed_objects),
        'failed_objects': int(failed_objects),
        'status': 'PASS' if failed_objects == 0 else 'FAIL',
        'meaning': meaning,
    })

PIPELINE_READINESS = pd.DataFrame(readiness_rows)
display(PIPELINE_READINESS)

# Database chưa được phép bàn giao nếu còn bất kỳ checkpoint FAIL nào.
failed_checkpoints = PIPELINE_READINESS.loc[
    PIPELINE_READINESS['status'] != 'PASS'
]
if not failed_checkpoints.empty:
    failed_names = ', '.join(failed_checkpoints['checkpoint'].tolist())
    raise RuntimeError(
        f'Pipeline chưa sẵn sàng bàn giao: {failed_names}'
    )

print('Tất cả checkpoint của Notebook 02 đều PASS.')

,checkpoint,checked_objects,passed_objects,failed_objects,status,meaning
0,input_files,5,5,0,PASS,"CSV tồn tại, không rỗng và đúng header Noteboo..."
1,staging_load,5,5,0,PASS,Row count và metadata staging khớp nguồn
2,fact_load,5,5,0,PASS,Fact row count khớp staging
3,analytical_views,5,5,0,PASS,Monthly views tồn tại và đúng schema
4,cross_level_joins,4,4,0,PASS,Không có row multiplication hoặc unexpected un...
5,aggregations,6,6,0,PASS,Aggregation grain và monthly coverage hợp lệ
6,indexes,13,13,0,PASS,"Index đúng bảng, cột và unique property"


Tất cả checkpoint của Notebook 02 đều PASS.


In [66]:
# Inventory contract liệt kê mọi object mà file backup phải mang sang máy Notebook 03.
EXPECTED_DATABASE_OBJECTS = {
    'staging_tables': {
        'staging_global', 'staging_country', 'staging_state',
        'staging_city', 'staging_major_city',
    },
    'dimension_tables': {
        'dim_date', 'dim_country', 'dim_state', 'dim_city',
    },
    'fact_tables': {
        'fact_global_temperature', 'fact_country_temperature',
        'fact_state_temperature', 'fact_city_temperature',
        'fact_major_city_temperature',
    },
    'monthly_views': {
        'vw_global_temperature', 'vw_country_temperature',
        'vw_state_temperature', 'vw_city_temperature',
        'vw_major_city_temperature',
    },
    'materialized_views': {
        'mv_global_temperature_yearly',
        'mv_global_temperature_decadal',
        'mv_country_temperature_yearly',
        'mv_state_temperature_yearly',
        'mv_city_temperature_yearly',
        'mv_major_city_temperature_yearly',
    },
    'pipeline_indexes': set(EXPECTED_INDEXES),
}

# Manifest mô tả monthly view, grain, lineage và cột nhiệt độ cho từng dataset.
HANDOFF_VIEW_CONFIG = {
    'global': {
        'view': 'vw_global_temperature', 'grain': 'date_id',
        'temperature_columns': (
            'land_average_temperature',
            'land_and_ocean_average_temperature',
        ),
    },
    'country': {
        'view': 'vw_country_temperature',
        'grain': 'date_id + country_id',
        'temperature_columns': ('average_temperature',),
    },
    'state': {
        'view': 'vw_state_temperature',
        'grain': 'date_id + state_id',
        'temperature_columns': ('average_temperature',),
    },
    'city': {
        'view': 'vw_city_temperature',
        'grain': 'date_id + city_id',
        'temperature_columns': ('average_temperature',),
    },
    'major_city': {
        'view': 'vw_major_city_temperature',
        'grain': 'date_id + city_id',
        'temperature_columns': ('average_temperature',),
    },
}

# Thu thập riêng kiểm kê object và data contract bàn giao.
inventory_rows = []
handoff_rows = []

try:
    with closing(psycopg2.connect(**DB_CONFIG)) as connection:
        connection.set_session(readonly=True, autocommit=True)
        with connection.cursor() as cursor:
            # Final audit chỉ đọc catalog/row count và có timeout để tránh treo notebook.
            cursor.execute("SET lock_timeout = '5s'")
            cursor.execute("SET statement_timeout = '120s'")

            cursor.execute(
                """
                SELECT table_name FROM information_schema.tables
                WHERE table_schema = 'public'
                  AND table_type = 'BASE TABLE';
                """
            )
            actual_tables = {row[0] for row in cursor.fetchall()}
            cursor.execute(
                """
                SELECT table_name FROM information_schema.views
                WHERE table_schema = 'public';
                """,
            )
            actual_views = {row[0] for row in cursor.fetchall()}
            cursor.execute(
                """
                SELECT matviewname FROM pg_matviews
                WHERE schemaname = 'public';
                """
            )
            actual_materialized_views = {row[0] for row in cursor.fetchall()}
            cursor.execute(
                """
                SELECT indexname FROM pg_indexes
                WHERE schemaname = 'public';
                """
            )
            actual_indexes = {row[0] for row in cursor.fetchall()}

            # Cùng tập tables được lọc tiếp bằng expected set cho staging/dimension/fact.
            actual_by_group = {
                'staging_tables': actual_tables,
                'dimension_tables': actual_tables,
                'fact_tables': actual_tables,
                'monthly_views': actual_views,
                'materialized_views': actual_materialized_views,
                'pipeline_indexes': actual_indexes,
            }
            # So sánh expected với catalog và ghi rõ object còn thiếu.
            for object_group, expected_objects in EXPECTED_DATABASE_OBJECTS.items():
                missing_objects = expected_objects - actual_by_group[object_group]
                found_count = len(expected_objects) - len(missing_objects)
                inventory_rows.append({
                    'object_group': object_group,
                    'expected_count': len(expected_objects),
                    'found_count': found_count,
                    'missing_objects': ', '.join(sorted(missing_objects)),
                    'status': 'PASS' if not missing_objects else 'FAIL',
                })

            # Row count monthly view phải bảo toàn số dòng nguồn trước khi cleaning.
            for dataset_name, config in HANDOFF_VIEW_CONFIG.items():
                cursor.execute(
                    sql.SQL('SELECT COUNT(*) FROM {}').format(
                        sql.Identifier(config['view'])
                    )
                )
                actual_rows = int(cursor.fetchone()[0])
                expected_rows = int(SOURCE_ROW_COUNTS[dataset_name])
                handoff_rows.append({
                    'dataset': dataset_name,
                    'source_view': config['view'],
                    'grain': config['grain'],
                    'lineage_column': 'source_staging_id',
                    'temperature_columns': ', '.join(
                        config['temperature_columns']
                    ),
                    'expected_rows': expected_rows,
                    'actual_rows': actual_rows,
                    'status': (
                        'PASS' if actual_rows == expected_rows else 'FAIL'
                    ),
                })
except psycopg2.Error as error:
    raise RuntimeError('Final database audit thất bại.') from error

# Hai DataFrame này là biên bản bàn giao có thể đối chiếu sau restore.
DATABASE_OBJECT_INVENTORY = pd.DataFrame(inventory_rows)
HANDOFF_MANIFEST = pd.DataFrame(handoff_rows)
display(DATABASE_OBJECT_INVENTORY)
display(HANDOFF_MANIFEST)

inventory_failed = (DATABASE_OBJECT_INVENTORY['status'] != 'PASS').any()
handoff_failed = (HANDOFF_MANIFEST['status'] != 'PASS').any()
if inventory_failed or handoff_failed:
    raise RuntimeError(
        'Database chưa đạt điều kiện bàn giao. Xem inventory/manifest ở trên.'
    )

# Cờ READY chỉ được đặt sau khi inventory và handoff manifest cùng PASS.
POSTGRESQL_PIPELINE_READY = True
print('POSTGRESQL PIPELINE: READY')
print(
    'Năm monthly analytical views đã sẵn sàng bàn giao cho '
    '03_data_cleaning.ipynb.'
)

,object_group,expected_count,found_count,missing_objects,status
0,staging_tables,5,5,,PASS
1,dimension_tables,4,4,,PASS
2,fact_tables,5,5,,PASS
3,monthly_views,5,5,,PASS
4,materialized_views,6,6,,PASS
5,pipeline_indexes,13,13,,PASS


,dataset,source_view,grain,lineage_column,temperature_columns,expected_rows,actual_rows,status
0,global,vw_global_temperature,date_id,source_staging_id,"land_average_temperature, land_and_ocean_avera...",3192,3192,PASS
1,country,vw_country_temperature,date_id + country_id,source_staging_id,average_temperature,577462,577462,PASS
2,state,vw_state_temperature,date_id + state_id,source_staging_id,average_temperature,645675,645675,PASS
3,city,vw_city_temperature,date_id + city_id,source_staging_id,average_temperature,5010113,5010113,PASS
4,major_city,vw_major_city_temperature,date_id + city_id,source_staging_id,average_temperature,239177,239177,PASS


POSTGRESQL PIPELINE: READY
Năm monthly analytical views đã sẵn sàng bàn giao cho 03_data_cleaning.ipynb.


### b. Backup và restore database cho người làm Notebook 03

Chỉ tạo bản backup sau khi final audit in `POSTGRESQL PIPELINE: READY`. PostgreSQL trên máy người tạo Notebook 02 là `localhost`; thành viên khác phải restore một bản sao trên PostgreSQL cục bộ của họ và không dùng `.env` hay mật khẩu của người gửi.

#### Bước 1 — Người làm Notebook 02 tạo backup `.sql` bằng pgAdmin 4

1. Mở pgAdmin 4 và kết nối PostgreSQL trên máy đang chứa pipeline.
2. Trong Browser, mở `Servers` → server local → `Databases`.
3. Nhấp chuột phải đúng database `climate_db` → **Backup…**. Không chọn riêng schema hoặc table vì cần backup toàn bộ database.
4. Trong tab **General**:
   - **Filename:** chọn đường dẫn đầy đủ, ví dụ `D:\handoff\climate_db_notebook02_complete.sql`.
   - **Format:** `Plain`.
   - **Encoding:** `UTF8`.
   - **Role name:** chọn `postgres` hoặc role đang có quyền đọc toàn bộ database.
5. Trong tab **Data Options**:
   - Phần **Sections:** bật `Pre-data`, `Data` và `Post-data` để lấy đủ schema, dữ liệu, indexes và constraints.
   - Phần **Type of objects:** để `Only data = No` và `Only schemas = No`.
   - Phần **Do not save:** đặt `Owner = Yes`, `Privileges = Yes` và `Tablespaces = Yes` để giảm lỗi khác user/tablespace khi chuyển máy.
   - Không loại `Comments`, vì comments là một phần tài liệu của database.
6. Trong tab **Query Options**:
   - `Use INSERT commands = No` để pg_dump dùng `COPY`, restore nhanh hơn.
   - `Include CREATE DATABASE statement = No` vì người nhận sẽ chủ động tạo database rỗng.
   - `Include DROP DATABASE statement = No` để file không tự xóa database đích.
7. Trong tab **Objects**, không chọn một danh sách con; giữ toàn bộ database trong phạm vi backup.
8. Nhấn **Backup**. Theo dõi job trong tab **Processes** và chỉ sử dụng file khi trạng thái kết thúc thành công, không có error.

File Plain chứa các câu lệnh SQL để tái tạo schema, data, views, materialized views, constraints và indexes. Vì database có City hơn 5 triệu dòng, file có thể lớn và quá trình backup cần thời gian; chia sẻ file ngoài Git. Sau khi pgAdmin hoàn tất, tạo checksum trong PowerShell:

```powershell
Get-FileHash "climate_db_notebook02_complete.sql" -Algorithm SHA256
```

Gửi file `.sql` cùng mã SHA256; không gửi `.env`. Tham khảo giao diện chính thức tại [pgAdmin 4 — Backup Dialog](https://www.pgadmin.org/docs/pgadmin4/latest/backup_dialog.html).

#### Bước 2 — Người làm Notebook 03 tạo database rỗng bằng pgAdmin 4

1. Cài PostgreSQL cùng major version (khuyến nghị 18) hoặc phiên bản mới hơn tương thích, sau đó mở pgAdmin 4.
2. Kết nối server PostgreSQL local của người nhận.
3. Trong Browser, nhấp chuột phải `Databases` → **Create** → **Database…**.
4. Trong tab **General**, nhập **Database = `climate_db`** và chọn **Owner = `postgres`** hoặc role local có quyền tạo object.
5. Nhấn **Save** và xác nhận database mới đang rỗng.

Nếu `climate_db` đã tồn tại và có dữ liệu, nên tạo database rỗng với tên khác thay vì restore đè. Khi đổi tên, cập nhật `DB_NAME` tương ứng trong `.env`.

#### Bước 3 — Restore file Plain `.sql` bằng pgAdmin 4

1. Nếu nhóm sử dụng checksum, kiểm tra SHA256 của file nhận được bằng công cụ checksum trên máy và so sánh với mã người gửi trước khi restore.
2. Trong Browser, nhấp chuột phải database rỗng `climate_db` → **Restore…**.
3. Trong tab **General**:
   - **Format:** chọn `Plain`.
   - **Filename:** chọn file `climate_db_notebook02_complete.sql` đã nhận.
   - **Role name:** chọn `postgres` hoặc owner của database đích.
4. Với định dạng Plain, các tùy chọn restore khác không áp dụng; không chọn restore từng table riêng lẻ.
5. Nhấn **Restore**. pgAdmin thực thi script Plain ở backend bằng `psql` với cơ chế hạn chế lệnh nguy hiểm.
6. Theo dõi tab **Processes** và chỉ tiếp tục khi job kết thúc thành công, không có error. Sau đó chọn database → **Refresh** để thấy tables, views và materialized views.

Nếu danh sách **Format** không có `Plain`, hãy cập nhật pgAdmin 4 lên phiên bản hỗ trợ Plain restore; không đổi file `.sql` sang Custom bằng cách đổi phần mở rộng. Tham khảo [pgAdmin 4 — Restore Dialog](https://www.pgadmin.org/docs/pgadmin4/latest/restore_dialog.html).

#### Bước 4 — Tạo `.env` bằng VS Code trên máy Notebook 03

File `.env` chỉ lưu thông tin để Python kết nối tới bản PostgreSQL vừa restore trên **máy người làm Notebook 03**. Đây không phải file được lấy từ người làm Notebook 02 và tuyệt đối không dùng mật khẩu của người gửi.

**4.1. Tạo file đúng vị trí**

1. Người làm Notebook 03 mở **toàn bộ thư mục dự án** `Global-Surface-Temperature-Analysis` bằng VS Code qua **File → Open Folder…**; không chỉ mở riêng notebook.
2. Trong bảng **Explorer**, nhấp chuột phải ngay thư mục gốc của dự án → **New File**.
3. Đặt tên chính xác là `.env` rồi nhấn Enter. File phải nằm cùng cấp với các thư mục `notebooks`, `SQL` và `data`, theo cấu trúc:

```text
Global-Surface-Temperature-Analysis/
├── .env                  ← tạo tại đây
├── notebooks/
│   └── 03_data_cleaning.ipynb
├── SQL/
└── data/
```

Không tạo `.env` bên trong `notebooks`. Trên Windows cần kiểm tra file không bị đặt thành `.env.txt`; tạo trực tiếp bằng VS Code sẽ tránh lỗi này.

**4.2. Điền cấu hình PostgreSQL local của người nhận**

Dán nội dung sau vào `.env`, thay giá trị `DB_USER` và `DB_PASSWORD` bằng tài khoản mà người làm Notebook 03 dùng để đăng nhập PostgreSQL trên máy họ:

```env
DB_HOST=localhost
DB_PORT=5432
DB_NAME=climate_db
DB_USER=postgres
DB_PASSWORD=mat_khau_postgresql_tren_may_nguoi_lam_notebook_03
```

Ý nghĩa các biến:

- `DB_HOST=localhost`: PostgreSQL đang chạy trên chính máy của người làm Notebook 03.
- `DB_PORT=5432`: cổng PostgreSQL local; chỉ đổi nếu lúc cài đặt đã chọn cổng khác.
- `DB_NAME=climate_db`: tên database đích ở Bước 2. Nếu restore vào tên khác thì ghi tên đó.
- `DB_USER=postgres`: user PostgreSQL local hoặc role khác có quyền `CONNECT` và `SELECT` các view.
- `DB_PASSWORD=...`: mật khẩu của `DB_USER` trên máy người nhận. Nếu mật khẩu có khoảng trắng hoặc ký tự `#`, nên đặt toàn bộ giá trị trong dấu nháy kép, ví dụ `DB_PASSWORD="Mat khau#03"`.

Sau khi điền, nhấn `Ctrl+S`. Không thêm dấu phẩy, không viết theo cú pháp Python và không để khoảng trắng quanh dấu `=`. Với kết nối local thông thường không cần thêm `DB_SSLMODE`.

**4.3. Bảo vệ thông tin đăng nhập**

Kiểm tra `.gitignore` ở project root đã có dòng `.env`. Không commit, gửi qua chat, chụp màn hình hoặc đưa nội dung `.env` vào output notebook. Mỗi thành viên tự tạo file riêng bằng thông tin PostgreSQL trên máy mình.

**4.4. Kiểm tra Notebook 03 đọc đúng `.env`**

Trước phần làm sạch, Notebook 03 cần cài `python-dotenv` và `psycopg2-binary` trong đúng environment/kernel đang chạy. Sau đó có thể dùng cell sau để kiểm tra cấu hình và kết nối mà không hiển thị mật khẩu:

```python
import os
from pathlib import Path
from contextlib import closing

import psycopg2
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.env').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy .env ở thư mục gốc dự án.')


project_root = find_project_root(Path.cwd().resolve())
env_path = project_root / '.env'
load_dotenv(env_path, override=True)

required = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError('Thiếu biến trong .env: ' + ', '.join(missing))

db_config = {
    'host': os.environ['DB_HOST'],
    'port': int(os.environ['DB_PORT']),
    'dbname': os.environ['DB_NAME'],
    'user': os.environ['DB_USER'],
    'password': os.environ['DB_PASSWORD'],
    'connect_timeout': 10,
}

with closing(psycopg2.connect(**db_config)) as connection:
    with connection.cursor() as cursor:
        cursor.execute('SELECT current_database(), current_user;')
        database_name, database_user = cursor.fetchone()

print(f'Đã đọc cấu hình từ: {env_path}')
print(f'Kết nối thành công: database={database_name}, user={database_user}')
```

Kết quả hợp lệ phải cho biết `database=climate_db` (hoặc đúng tên database đã chọn) và đúng user local. Nếu báo `password authentication failed`, sửa `DB_USER`/`DB_PASSWORD`; nếu báo `connection refused`, kiểm tra PostgreSQL service và `DB_PORT`; nếu không tìm thấy `.env`, kiểm tra lại vị trí và tên file. Cell kiểm tra chỉ xác nhận kết nối; row count sau restore vẫn được kiểm tra ở Bước 5.

#### Bước 5 — Xác nhận restore bằng Query Tool của pgAdmin 4

1. Nhấp chuột phải `climate_db` → **Query Tool**.
2. Dán các truy vấn sau và nhấn **Execute/Refresh** hoặc `F5`:

```sql
SELECT COUNT(*) FROM vw_global_temperature;      -- 3,192
SELECT COUNT(*) FROM vw_country_temperature;     -- 577,462
SELECT COUNT(*) FROM vw_state_temperature;       -- 645,675
SELECT COUNT(*) FROM vw_city_temperature;        -- 5,010,113
SELECT COUNT(*) FROM vw_major_city_temperature;  -- 239,177
```

Chỉ bắt đầu `03_data_cleaning.ipynb` khi cả năm row count khớp `HANDOFF_MANIFEST`. Không chạy Mục 5 của Notebook 02 trên database đã restore. Notebook 03 đọc monthly views, giữ `source_staging_id`, ghi lại quy tắc cleaning và row count trước–sau; yearly materialized views chỉ dùng tham khảo cho EDA.

### Kết luận

Notebook 02 hoàn tất khi pipeline `READY`, backup có checksum và máy người nhận restore/kiểm tra row count thành công. Từ thời điểm đó, mọi thay đổi missing values, duplicates hoặc outliers thuộc trách nhiệm của Notebook 03 và phải được tài liệu hóa để vẫn truy vết về dữ liệu PostgreSQL ban đầu.